# B0 raw-input baseline: resume from update 2,999

This notebook continues the original raw-input residual U-Net for the remaining
7,001 of its planned 10,000 updates. It uses the same 218/49/101 patient split,
seed 1729 and 96-cubed patches as B5. B0 uses modality dropout without corruption
augmentation. Upload baseline_resume_input as a private Kaggle dataset and attach
it alongside the same BraTS20 dataset. Missing or mismatched input stops execution.

The shared tutorial retains DetailTrust component tests/profiling on disposable
models; the research experiment list contains B0 only. No B5 stage is retrained.
The test set stays sealed. B0 validation and robustness results are exported when
the selected checkpoint is evaluated. Its full selected checkpoint is under
experiments/B0_s1729/RAW/best.pt; B5-only export filenames are not produced.


## A — Dependencies and reproducibility
Install only missing libraries. Preserve Kaggle's existing CUDA-enabled PyTorch. No credentials enter this cell.

In [ ]:
import importlib.util
import subprocess
import sys
missing = [name for name in ("numpy", "scipy", "nibabel", "matplotlib", "nbformat")
           if importlib.util.find_spec(name) is None]
if missing:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "--quiet", *missing])
if importlib.util.find_spec("torch") is None:
    raise RuntimeError("Select Kaggle's GPU Python image with PyTorch installed.")

In [ ]:
from dataclasses import dataclass, asdict, replace
from pathlib import Path
import hashlib
import json
import math
import os
import random
import platform
import time
import shutil
import importlib.metadata
import numpy as np
import torch
from pathlib import Path, PurePosixPath
import re
from scipy import ndimage
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from contextlib import nullcontext
from torch import nn
from torch.utils.checkpoint import checkpoint
import itertools
from scipy.stats import spearmanr
from dataclasses import asdict
from dataclasses import asdict, replace
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import csv
from matplotlib.colors import ListedColormap, BoundaryNorm
from dataclasses import replace
import tempfile
import importlib.util

### Named settings and experiment registry
Every mode keeps widths 32/64/128/256/320. Updates are sampled patches; an epoch is a logging interval.

In [ ]:
MODALITIES = ("t1", "t1ce", "t2", "flair")

In [ ]:
REGIONS = {"WT": (1, 2, 3), "TC": (1, 3), "ET": (3,)}

In [ ]:
CLASSES = {"background": (0,), "NCR_NET": (1,), "edema": (2,), "ET_class": (3,)}

In [ ]:
LABEL_MAPPING = {0: 0, 1: 1, 2: 2, 4: 3}

In [ ]:
DATASET_SLUG = "awsaf49/brats20-dataset-training-validation"

In [ ]:
WIDTHS = (32, 64, 128, 256, 320)

**`RunConfig`** — An epoch is a reporting interval of sampled patches, not a volume pass.

In [ ]:
@dataclass
class RunConfig:
    """An epoch is a reporting interval of sampled patches, not a volume pass."""
    mode: str = "smoke"
    data_root: str = "/kaggle/input/brats20-dataset-training-validation"
    output_dir: str = "/kaggle/working/detailtrust"
    dataset_version: str | None = None
    seed: int = 1729
    seeds: tuple = (1729,)
    split_seed: int = 20261012
    patch: tuple = (96, 96, 96)
    batch_size: int = 1
    train_subjects: int = 4
    val_subjects: int | None = 1
    updates_a: int = 2
    updates_b: int = 2
    updates_c: int = 2
    epoch_updates: int = 2
    validation_every: int = 2
    precision: str = "amp_fp16"
    checkpoint_blocks: bool = True
    workers: int = 0
    lr: float = 2e-4
    gate_lr: float = 1e-3
    weight_decay: float = 1e-5
    warmup_updates: int = 100
    branch_dropout: float = 0.25
    utility_tau: float = 0.1
    utility_weight: float = 0.1
    corruption_probability: float = 0.5
    checkpoint_seconds: float = 1200
    max_hours: float = 1.5
    training_budget_fraction: float = 0.9
    profile_warmup: int = 1
    profile_updates: int = 2
    inference_repeats: int = 2
    early_stopping_patience: int | None = None
    surface_tolerance_mm: float = 1.0
    small_et_threshold_mm3: float | None = None
    nonzero_threshold: float = 1e-5
    variants: tuple = ("B5",)
    run_final_test: bool = False
    protocol_locked: bool = False
    robustness: str = "smoke"
    cache_gb: float = 4.0
    synthetic: bool = False
    resume_dir: str | None = None

**`configuration`** — Keep widths fixed in every mode; synthetic smoke alone uses 16³ patches.

In [ ]:
def configuration(mode: str = "smoke", **overrides) -> RunConfig:
    """Keep widths fixed in every mode; synthetic smoke alone uses 16³ patches."""
    presets = {
        "smoke": {},
        "pilot": dict(train_subjects=100, val_subjects=5, updates_a=400,
                      updates_b=450, updates_c=150, epoch_updates=50,
                      validation_every=100, profile_warmup=5, profile_updates=100,
                      max_hours=9, robustness="development", cache_gb=8),
        "main": dict(train_subjects=219, val_subjects=None, updates_a=4000,
                     updates_b=4500, updates_c=1500, epoch_updates=100,
                     validation_every=500, profile_warmup=5, profile_updates=100,
                     max_hours=9, robustness="complete", cache_gb=8),
        "extended": dict(train_subjects=219, val_subjects=None, updates_a=4000,
                         updates_b=4500, updates_c=1500, epoch_updates=100,
                         validation_every=500, profile_warmup=5, profile_updates=100,
                         max_hours=9, robustness="complete", cache_gb=8,
                         training_budget_fraction=0.7,
                         seeds=(1729, 1730, 1731),
                         variants=("B0", "B1", "B2", "B4", "B5", "B3", "A1",
                                   "A2", "A3", "A4", "A5", "A6", "A7")),
    }
    if mode not in presets:
        raise ValueError(f"Unknown mode {mode}")
    cfg = RunConfig(**(dict(mode=mode) | presets[mode] | overrides))
    assert cfg.batch_size == 1, "This audited pipeline specifies batch size one."
    assert all(p >= 16 and p % 16 == 0 for p in cfg.patch)
    assert cfg.workers == 0, "Zero workers preserves exact patch-stream resume."
    assert 0 < cfg.training_budget_fraction < 1
    if cfg.run_final_test:
        assert cfg.protocol_locked and mode in ("main", "extended")
        assert not cfg.synthetic
    return cfg

**`experiment_registry`** — All requested variants exist even when not selected for this allocation.

In [ ]:
def experiment_registry() -> dict:
    """All requested variants exist even when not selected for this allocation."""
    base = dict(split="learned", anchored=True, gate="spatial", utility=True,
                shuffle=False, joint=False, resolution_loss=False, corruption=True)
    variants = {
        "B0": ("Raw U-Net with modality dropout", dict(gate="raw", corruption=False)),
        "B1": ("Raw U-Net plus matched corruption", dict(gate="raw")),
        "B2": ("Anchored corrections, gates one", dict(gate="ones", utility=False)),
        "B3": ("Four global learned sigmoid scalars", dict(gate="global", utility=False)),
        "B4": ("Spatial gates, segmentation supervision only", dict(utility=False)),
        "B5": ("Full frozen-stage utility-supervised model", {}),
        "A1": ("Spatially permuted utility targets within each branch", dict(shuffle=True)),
        "A2": ("Fixed 0.8 mm Gaussian", dict(split="fixed")),
        "A3": ("Raw context and raw anchored refinement", dict(split="raw")),
        "A4": ("Haar low-pass reconstruction and residual", dict(split="haar")),
        "A5": ("Unanchored correction R(F,H,e)", dict(anchored=False)),
        "A6": ("Unfreeze all components in Stage C", dict(joint=True)),
        "A7": ("Known-resolution occupancy loss, weight 0.1", dict(resolution_loss=True)),
    }
    return {k: dict(description=d, **(base | v)) for k, (d, v) in variants.items()}

In [ ]:
def seed_everything(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.benchmark = False
    torch.backends.cudnn.deterministic = True
    torch.use_deterministic_algorithms(True, warn_only=True)

**`safe_json`** — Undefined metrics become null; their reason belongs in the metric record.

In [ ]:
def safe_json(value):
    """Undefined metrics become null; their reason belongs in the metric record."""
    if isinstance(value, dict):
        return {str(k): safe_json(v) for k, v in value.items()}
    if isinstance(value, (list, tuple)):
        return [safe_json(v) for v in value]
    if isinstance(value, (Path, torch.device)):
        return str(value)
    if isinstance(value, np.ndarray):
        return safe_json(value.tolist())
    if isinstance(value, np.generic):
        return safe_json(value.item())
    if isinstance(value, torch.Tensor):
        return safe_json(value.detach().cpu().tolist())
    if isinstance(value, float) and not math.isfinite(value):
        return None
    return value

**`write_json`** — Atomic replacement avoids half-written reports after interruption.

In [ ]:
def write_json(path, value) -> None:
    """Atomic replacement avoids half-written reports after interruption."""
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    tmp = path.with_suffix(path.suffix + ".tmp")
    tmp.write_text(json.dumps(safe_json(value), indent=2, sort_keys=True,
                              allow_nan=False), encoding="utf-8")
    json.loads(tmp.read_text(encoding="utf-8"))
    tmp.replace(path)

In [ ]:
def file_hash(path) -> str:
    digest = hashlib.sha256()
    with open(path, "rb") as handle:
        for block in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(block)
    return digest.hexdigest()

**`environment_record`** — Whitelist runtime information; never capture environment variables.

In [ ]:
def environment_record(output_dir) -> dict:
    """Whitelist runtime information; never capture environment variables."""
    versions = {}
    for name in ("torch", "numpy", "scipy", "nibabel", "matplotlib", "nbformat"):
        try:
            versions[name] = importlib.metadata.version(name)
        except importlib.metadata.PackageNotFoundError:
            versions[name] = None
    return dict(python=platform.python_version(), platform=platform.platform(),
                packages=versions, cuda=torch.version.cuda,
                gpu_name=torch.cuda.get_device_name(0) if torch.cuda.is_available() else None,
                gpu_count=torch.cuda.device_count(),
                gpu_memory_bytes=torch.cuda.get_device_properties(0).total_memory
                if torch.cuda.is_available() else None,
                free_disk_bytes=shutil.disk_usage(output_dir).free)

### Verified continuation
The uploader attaches a private checkpoint input and sets its exact bundle ID. A missing or incompatible checkpoint stops execution; it never falls back to fresh weights.

**`restore_resume_bundle`** — Missing, corrupted, ambiguous or incompatible inputs stop execution.

In [ ]:
def restore_resume_bundle(bundle_id, output_dir, cfg, input_root='/kaggle/input'):
    """Missing, corrupted, ambiguous or incompatible inputs stop execution."""
    matches = []
    for path in Path(input_root).rglob('resume_manifest.json'):
        manifest = json.loads(path.read_text())
        if manifest.get('bundle_id') == bundle_id:
            matches.append((path, manifest))
    if len(matches) != 1:
        raise RuntimeError(f'Expected one attached resume bundle {bundle_id}; found {len(matches)}. Refusing fresh training.')
    manifest_path, manifest = matches[0]
    canonical = {k: v for k, v in manifest.items() if k != 'bundle_id'}
    assert hashlib.sha256(json.dumps(canonical, sort_keys=True).encode()).hexdigest() == bundle_id, 'Resume manifest hash mismatch'
    destination = Path(output_dir).resolve()
    entries = {}
    for entry in manifest['files']:
        relative = PurePosixPath(entry['path'])
        assert not relative.is_absolute() and '..' not in relative.parts and '\\' not in entry['path'] and ':' not in entry['path']
        payload = entry['payload']
        assert Path(payload).name == payload and '/' not in payload and '\\' not in payload
        source = manifest_path.parent/payload
        assert source.is_file() and source.stat().st_size == entry['bytes'], f'Missing/incomplete resume file: {relative}'
        assert file_hash(source) == entry['sha256'], f'Resume hash mismatch: {relative}'
        assert entry['path'] not in entries, 'Duplicate resume path'
        entries[entry['path']] = source
    old = json.loads(entries['experiment_config.json'].read_text())
    # Other checks, including stage fingerprints, run again before optimizer updates.
    keys = ('mode', 'seed', 'seeds', 'split_seed', 'patch', 'batch_size', 'train_subjects',
            'val_subjects', 'updates_a', 'updates_b', 'updates_c', 'validation_every', 'precision',
            'lr', 'gate_lr', 'warmup_updates', 'weight_decay', 'branch_dropout', 'utility_tau',
            'utility_weight', 'corruption_probability', 'synthetic', 'variants')
    for key in keys:
        assert json.dumps(old[key]) == json.dumps(getattr(cfg, key)), f'Resume setting changed: {key}; refusing fresh training'
    for path, expected in manifest['progress'].items():
        assert path in entries, f'Required checkpoint missing: {path}'
        state = torch.load(entries[path], map_location='cpu', weights_only=False)
        assert state['update'] == expected['update'] and state['stage'] == expected['stage'], 'Checkpoint progress differs from history'
        assert all(key in state for key in ('optimizer_state', 'scheduler_state', 'scaler_state', 'rng', 'fingerprint'))
        del state
    for relative, source in entries.items():
        target = (destination/relative).resolve()
        assert target.is_relative_to(destination)
        target.parent.mkdir(parents=True, exist_ok=True)
        if target.exists():
            # Rerunning this cell must never overwrite newer in-session progress.
            raise RuntimeError(f'Resume destination already contains {relative}. Start a clean session rather than overwrite progress.')
        shutil.copy2(source, target)
    restored = dict(bundle_id=bundle_id, source=str(manifest_path.parent), progress=manifest['progress'],
                    verified_files=len(entries), status='restored')
    write_json(destination/'resume_restore.json', restored)
    print('Restored saved training progress:', manifest['progress'], flush=True)
    return restored

In [ ]:
# The main settings you edit before Save Version / Run All.
MODE = "main"
EXPERIMENTS = ("B0",)  # recovered raw-input baseline only
SYNTHETIC_TEST_ONLY = False    # True ONLY for pipeline debugging, never research
RUN_FINAL_TEST = False        # keep False during development
PROTOCOL_LOCKED = False       # must be True before final test evaluation
OUTPUT_DIRECTORY = "/kaggle/working/detailtrust"
RESUME_DIRECTORY = None
RESUME_BUNDLE_ID = "d86a83a51ed6489e4aa6f7d19731a942ab061fc37dc5cd139d99e5003a1cb19c"
cfg = configuration(MODE, variants=EXPERIMENTS, output_dir=OUTPUT_DIRECTORY,
                    synthetic=SYNTHETIC_TEST_ONLY, resume_dir=RESUME_DIRECTORY,
                    run_final_test=RUN_FINAL_TEST, protocol_locked=PROTOCOL_LOCKED)
if SYNTHETIC_TEST_ONLY:
    cfg = replace(cfg, patch=(16, 16, 16), precision="float32", profile_updates=1,
                  profile_warmup=0, max_hours=1, val_subjects=1)
seed_everything(cfg.seed)
print({k: getattr(cfg, k) for k in ("mode", "patch", "train_subjects", "variants", "precision")})

In [ ]:
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
if device.type != "cuda" and not SYNTHETIC_TEST_ONLY:
    raise RuntimeError("Enable a Kaggle GPU for real-data training. CPU is supported only for explicit synthetic tests.")
torch.set_num_threads(min(4, os.cpu_count() or 1))
print("Training device:", device)
if device.type == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))
    print("Memory GiB:", torch.cuda.get_device_properties(0).total_memory / 2**30)

In [ ]:
output_dir = Path(cfg.output_dir)
output_dir.mkdir(parents=True, exist_ok=True)
(output_dir / "figures").mkdir(exist_ok=True)
if RESUME_BUNDLE_ID:
    restore_resume_bundle(RESUME_BUNDLE_ID, output_dir, cfg)
if cfg.resume_dir:
    previous = Path(cfg.resume_dir)
    assert previous.is_dir() and previous.resolve() != output_dir.resolve()
    for name in ("experiments",):
        if (previous/name).exists():
            shutil.copytree(previous/name, output_dir/name, dirs_exist_ok=True)
    for name in ("split_manifest.json",):
        if (previous/name).exists():
            shutil.copy2(previous/name, output_dir/name)
write_json(output_dir / "experiment_config.json", asdict(cfg))

In [ ]:
runtime_environment = environment_record(output_dir)
write_json(output_dir / "environment.json", runtime_environment)
print(runtime_environment)

In [ ]:
# A failed early data check still leaves an honest status artifact.
write_json(output_dir / "detailtrust_report.json", {
    "schema_version": "1.0.0", "experiment_status": "initializing",
    "training_configuration": asdict(cfg), "environment": runtime_environment,
    "validation_metrics": None, "test_metrics": None,
    "limitations": ["No experiment completed yet; final report replaces this initialization record."]})

## B — Dataset and preprocessing
    Match subjects before splitting. Original labels 0/1/2/4 map to internal 0/1/2/3.
    NiBabel reorients arrays to RAS without interpolating labels; the adapter checks
    1 mm spacing and affine alignment. A different spacing raises an error rather
    than silently changing the physical Gaussian scale. Crop boundaries are based
    only on available MRI. Tumour labels are used only for training-patch sampling
    and supervision. Each patch includes a four-voxel filtering halo.

**`discover_dataset_root`** — Use mounted data only. Ambiguous sources require the visible path setting.

In [ ]:
def discover_dataset_root(override=None, mount="/kaggle/input"):
    """Use mounted data only. Ambiguous sources require the visible path setting."""
    if override:
        root = Path(override)
        if not root.is_dir():
            raise FileNotFoundError(f"Configured DATASET_ROOT does not exist: {root}")
        return root
    mount = Path(mount)
    candidates = set()
    for path in mount.rglob("BraTS20_Training_*_t1.nii*"):
        relative = path.relative_to(mount)
        depth = 3 if relative.parts[0] == "datasets" else 1
        candidates.add(mount.joinpath(*relative.parts[:depth]))
    if len(candidates) != 1:
        raise ValueError(f"Found {len(candidates)} compatible mounts: {sorted(map(str, candidates))}. Set DATASET_ROOT explicitly; attach the NIfTI mirror first.")
    return next(iter(candidates))

**`discover_subjects`** — Group exact subject IDs. Unlabelled challenge validation is inventoried only.

In [ ]:
def discover_subjects(root):
    """Group exact subject IDs. Unlabelled challenge validation is inventoried only."""
    root = Path(root)
    if not root.exists():
        raise FileNotFoundError(f"Attach the NIfTI BraTS dataset; missing directory: {root}")
    groups, unmatched = {}, []
    pattern = re.compile(r"^(BraTS20_(?:Training|Validation)_\d+)_(t1ce|t1|t2|flair|seg)\.nii(?:\.gz)?$", re.I)
    for path in sorted(root.rglob("*.nii*")):
        match = pattern.match(path.name)
        if not match:
            unmatched.append(str(path.relative_to(root)))
            continue
        subject, modality = match.groups()
        group = groups.setdefault(subject, {"id": subject, "patient_id": subject, "files": {}})
        if modality.lower() in group["files"]:
            raise ValueError(f"Duplicate {subject}/{modality}; choose a single dataset version/root.")
        group["files"][modality.lower()] = str(path)
    eligible, excluded = [], []
    for subject in sorted(groups):
        record = groups[subject]
        missing = [m for m in MODALITIES if m not in record["files"]]
        if missing or "seg" not in record["files"]:
            excluded.append(dict(id=subject, missing_modalities=missing,
                                 reason="missing modalities" if missing else "no distributed label"))
        elif "_Training_" not in subject:
            excluded.append(dict(id=subject, reason="not in labelled training collection"))
        else:
            eligible.append(record)
    if not eligible:
        raise ValueError("No complete labelled BraTS20 NIfTI subjects found; HDF5/slice mirrors are incompatible.")
    return eligible, dict(discovered=len(groups), eligible=len(eligible), excluded=excluded,
                          unmatched_nifti=unmatched, adapter="BraTS20 exact filenames; no silent rename")

**`remap_labels`** — Original 0/1/2/4 -> exclusive 0/1/2/3; fail on fractional/unknown labels.

In [ ]:
def remap_labels(labels):
    """Original 0/1/2/4 -> exclusive 0/1/2/3; fail on fractional/unknown labels."""
    values = np.unique(labels)
    if not np.isfinite(values).all() or not set(values.tolist()) <= set(LABEL_MAPPING):
        raise ValueError(f"Unexpected BraTS20 label values: {values.tolist()}")
    result = np.zeros(labels.shape, np.uint8)
    for old, new in LABEL_MAPPING.items():
        result[labels == old] = new
    return result

**`load_subject`** — Losslessly reorient to RAS; assert aligned orthogonal 1 mm geometry.

No silent resampling. This adapter deliberately rejects a different spacing
release; GaussianSplit itself supports physical sigma for other spacings.
Array axes remain R/A/S and are called D/H/W by PyTorch (no axis transpose).

In [ ]:
def load_subject(record):
    """Losslessly reorient to RAS; assert aligned orthogonal 1 mm geometry.

    No silent resampling. This adapter deliberately rejects a different spacing
    release; GaussianSplit itself supports physical sigma for other spacings.
    Array axes remain R/A/S and are called D/H/W by PyTorch (no axis transpose).
    """
    import nibabel as nib
    loaded = {name: nib.load(path) for name, path in record["files"].items()}
    original = loaded["t1"]
    canonical = {name: nib.as_closest_canonical(img) for name, img in loaded.items()}
    reference = canonical["t1"]
    spacing = np.asarray(reference.header.get_zooms()[:3], float)
    if not np.allclose(spacing, 1.0, atol=1e-4):
        raise ValueError(f"{record['id']}: spacing {spacing}; configure an explicit resampling protocol.")
    directions = reference.affine[:3, :3] / spacing
    if not np.allclose(directions.T @ directions, np.eye(3), atol=1e-4):
        raise ValueError("Sheared voxel grid is unsupported for physical distance metrics.")
    for name, img in canonical.items():
        if img.shape != reference.shape or not np.allclose(img.affine, reference.affine, atol=1e-4):
            raise ValueError(f"{record['id']}: {name} is not spatially aligned.")
    images = np.stack([canonical[m].get_fdata(dtype=np.float32) for m in MODALITIES])
    if not np.isfinite(images).all():
        raise ValueError(f"{record['id']}: nonfinite MRI intensities")
    labels = remap_labels(canonical["seg"].get_fdata(dtype=np.float32))
    metadata = dict(id=record["id"], spacing=spacing.tolist(), canonical_shape=list(labels.shape),
                    canonical_affine=reference.affine.tolist(), original_shape=list(original.shape),
                    original_affine=original.affine.tolist(), orientation="RAS", array_order="R,A,S",
                    raw_labels=sorted(int(v) for v in np.unique(canonical["seg"].dataobj)))
    return images, labels, metadata

**`inventory_geometry`** — Audit every labelled case; failures stop training instead of being hidden.

In [ ]:
def inventory_geometry(records, output_path):
    """Audit every labelled case; failures stop training instead of being hidden."""
    rows = []
    for record in records:
        try:
            images, labels, metadata = load_subject(record)
            metadata["status"] = "passed"
            metadata["foreground_voxels"] = [int((x != 0).sum()) for x in images]
            metadata["et_volume_mm3"] = float((labels == 3).sum() * np.prod(metadata["spacing"]))
            # Hash image arrays for exact duplicate detection without label-based features.
            metadata["image_sha256"] = hashlib.sha256(images.tobytes()).hexdigest()
            metadata["file_sizes"] = {m: Path(p).stat().st_size for m, p in record["files"].items()}
            rows.append(metadata)
        except (ValueError, OSError) as error:
            rows.append(dict(id=record["id"], status="failed", reason=str(error)))
        write_json(output_path, rows)
    failed = [r for r in rows if r["status"] != "passed"]
    hashes = [r["image_sha256"] for r in rows if r["status"] == "passed"]
    if failed or len(hashes) != len(set(hashes)):
        raise ValueError("Dataset audit failed or duplicate image volumes exist; inspect dataset_integrity.json.")
    return rows

**`make_split`** — Custom subject split. At 369 cases use 219/50/100; never call it official.

In [ ]:
def make_split(records, seed):
    """Custom subject split. At 369 cases use 219/50/100; never call it official."""
    ids = sorted(r["id"] for r in records)
    assert len(ids) == len(set(ids)) and len(ids) >= 6
    shuffled = np.random.default_rng(seed).permutation(ids).tolist()
    if len(ids) == 369:
        n_train, n_val = 219, 50
    else:
        n_train, n_val = max(2, int(len(ids) * 219 / 369)), max(1, int(len(ids) * 50 / 369))
    split = dict(train=shuffled[:n_train], validation=shuffled[n_train:n_train + n_val],
                 test=shuffled[n_train + n_val:], seed=seed,
                 protocol="custom patient split of labelled BraTS20 training collection",
                 official=False, published_author_ids_recovered=False)
    assert_split(split)
    split["sha256"] = hashlib.sha256(json.dumps(split, sort_keys=True).encode()).hexdigest()
    return split

In [ ]:
def assert_split(split):
    parts = [set(split[k]) for k in ("train", "validation", "test")]
    assert all(parts), "Each split must contain at least one subject."
    assert all(len(split[k]) == len(set(split[k])) for k in ("train", "validation", "test"))
    assert not (parts[0] & parts[1] or parts[0] & parts[2] or parts[1] & parts[2])

**`normalize_foreground`** — Per-image nonzero foreground z-score; input features never use labels.

In [ ]:
def normalize_foreground(images):
    """Per-image nonzero foreground z-score; input features never use labels."""
    result = np.zeros_like(images, dtype=np.float32)
    for modality, image in enumerate(images):
        mask = image != 0
        values = image[mask]
        if not len(values) or values.std() < 1e-6:
            raise ValueError(f"Empty or nearly constant modality {MODALITIES[modality]}")
        result[modality, mask] = (values - values.mean()) / values.std()
    return result

**`image_crop`** — Determine crop solely from available MRI foreground, including at evaluation.

In [ ]:
def image_crop(images, available, margin=8):
    """Determine crop solely from available MRI foreground, including at evaluation."""
    foreground = np.any(images[np.asarray(available, bool)] != 0, axis=0)
    if not foreground.any():
        raise ValueError("No image foreground")
    coordinates = np.where(foreground)
    bounds = [(max(0, int(c.min()) - margin), min(n, int(c.max()) + margin + 1))
              for c, n in zip(coordinates, foreground.shape)]
    return tuple(slice(a, b) for a, b in bounds)

**`prepare_subject`** — Cache normalized full volumes; condition-specific cropping happens later.

In [ ]:
def prepare_subject(images, labels, metadata):
    """Cache normalized full volumes; condition-specific cropping happens later."""
    normalized = normalize_foreground(images)
    assert normalized.shape[1:] == labels.shape and labels.dtype == np.uint8
    return normalized, labels, metadata

**`SubjectStore`** — Bounded disk cache with mmap reads; no full dataset in RAM or GPU memory.

In [ ]:
class SubjectStore:
    """Bounded disk cache with mmap reads; no full dataset in RAM or GPU memory."""
    def __init__(self, records, cache_dir, max_gb=4.0):
        self.records = {r["id"]: r for r in records}
        self.directory = Path(cache_dir)
        self.directory.mkdir(parents=True, exist_ok=True)
        self.max_bytes = int(max_gb * 1e9)

    def load(self, subject):
        record = self.records[subject]
        signature = [(m, str(p), Path(p).stat().st_size, Path(p).stat().st_mtime_ns)
                     for m, p in sorted(record["files"].items())]
        key = hashlib.sha256(json.dumps(signature).encode()).hexdigest()[:16]
        stem = self.directory / (subject + "_" + key)
        xp, yp, mp = Path(str(stem) + "_x.npy"), Path(str(stem) + "_y.npy"), Path(str(stem) + ".json")
        if xp.exists() and yp.exists() and mp.exists():
            return np.load(xp, mmap_mode="r"), np.load(yp, mmap_mode="r"), json.loads(mp.read_text())
        images, labels, metadata = prepare_subject(*load_subject(record))
        used = sum(p.stat().st_size for p in self.directory.glob("*.*"))
        if used + images.nbytes + labels.nbytes < self.max_bytes:
            np.save(xp, images)
            np.save(yp, labels)
            write_json(mp, metadata)
        return images, labels, metadata

In [ ]:
def sample_availability(rng):
    mask = 15 if rng.random() < 0.25 else int(rng.integers(1, 15))
    return np.asarray([(mask >> i) & 1 for i in range(4)], np.float32)

**`extract_patch`** — Half class-balanced tumour centres, half available-image foreground.

In [ ]:
def extract_patch(images, labels, available, patch, rng, halo=4):
    """Half class-balanced tumour centres, half available-image foreground."""
    classes = [k for k in (1, 2, 3) if np.any(labels == k)]
    if classes and rng.random() < 0.5:
        candidates = np.argwhere(labels == rng.choice(classes))
    else:
        candidates = np.argwhere(np.any(images[available.astype(bool)] != 0, axis=0))
    centre = candidates[rng.integers(len(candidates))]
    starts = [int(c) - n // 2 for c, n in zip(centre, patch)]
    # Reflect at volume borders just as the Gaussian uses reflection.
    pads = [(max(0, halo - s), max(0, s + n + halo - dim))
            for s, n, dim in zip(starts, patch, labels.shape)]
    padded = np.pad(np.asarray(images), [(0, 0)] + pads, mode="reflect")
    ypad = np.pad(np.asarray(labels), pads, mode="constant")
    slices = tuple(slice(s + p[0] - halo, s + p[0] + n + halo)
                   for s, p, n in zip(starts, pads, patch))
    x = padded[(slice(None),) + slices].copy()
    y = ypad[slices][halo:-halo, halo:-halo, halo:-halo].copy()
    return x, y

**`corrupt_image`** — Image-space simulations. Mask generation never receives a segmentation.

In [ ]:
def corrupt_image(image, family, severity, rng, local=True, axis=2):
    """Image-space simulations. Mask generation never receives a segmentation."""
    image = np.asarray(image, np.float32)
    coarse_shape = None
    if family == "resolution":
        sigma = [0., 0., 0.]
        sigma[axis] = 0.5 * severity
        blurred = ndimage.gaussian_filter(image, sigma, mode="reflect")
        coarse_shape = list(image.shape)
        coarse_shape[axis] = max(2, int(round(image.shape[axis] / severity)))
        t = torch.from_numpy(blurred.copy())[None, None]
        # Adaptive averages define the exact positive row-normalized A_s for A7.
        coarse = F.adaptive_avg_pool3d(t, coarse_shape)
        changed = F.interpolate(coarse, image.shape, mode="trilinear", align_corners=False)[0, 0].numpy()
    elif family == "noise":
        changed = image + rng.normal(0, severity, image.shape).astype(np.float32)
    elif family == "bias":
        grid = rng.normal(0, severity, (4, 4, 4)).astype(np.float32)
        field = F.interpolate(torch.from_numpy(grid)[None, None], image.shape, mode="trilinear", align_corners=False)[0, 0].numpy()
        changed = image * np.clip(np.exp(field), 0.8, 1.2)
    elif family == "ghost":
        changed = image + severity * np.roll(image, image.shape[axis] // 4, axis=axis)
    elif family == "ringing":
        spectrum = np.fft.fft(image, axis=axis)
        frequency = np.abs(np.fft.fftfreq(image.shape[axis]))
        shape = [1, 1, 1]
        shape[axis] = -1
        spectrum *= (frequency <= (0.5 * (1 - severity))).reshape(shape)
        changed = np.fft.ifft(spectrum, axis=axis).real.astype(np.float32)
    else:
        raise ValueError(f"Unknown corruption {family}")
    if local:
        axes = np.ogrid[tuple(slice(0, n) for n in image.shape)]
        centre = [rng.uniform(0.2, 0.8) * n for n in image.shape]
        distance = sum(((a - c) / (0.22 * n)) ** 2 for a, c, n in zip(axes, centre, image.shape))
        mask = np.exp(-0.5 * distance).astype(np.float32)
        changed = image * (1 - mask) + changed * mask
    changed[image == 0] = 0
    return changed.astype(np.float32), coarse_shape

**`training_augmentation`** — Aligned flips are interpolation-free; intensity damage affects one scan.

In [ ]:
def training_augmentation(images, labels, available, rng, corruption_probability):
    """Aligned flips are interpolation-free; intensity damage affects one scan."""
    images, labels = images.copy(), labels.copy()
    for axis in range(3):
        if rng.random() < 0.5:
            images = np.flip(images, axis + 1).copy()
            labels = np.flip(labels, axis).copy()
    resolution = np.zeros(3, dtype=np.int64)
    if rng.random() < corruption_probability:
        modality = int(rng.choice(np.flatnonzero(available)))
        family = str(rng.choice(["resolution", "noise", "bias"]))
        severity = rng.uniform(1.5, 3) if family == "resolution" else rng.uniform(0.05, 0.15)
        axis = int(rng.integers(3))
        local = bool(rng.integers(2))
        images[modality], coarse = corrupt_image(images[modality], family, severity, rng, local, axis)
        # A7 applies only to global resolution damage; local mixtures do not share one A_s.
        if coarse and not local:
            resolution = np.asarray(labels.shape, np.int64)
            resolution[axis] = max(2, int(round(labels.shape[axis] / severity)))
    return images * available[:, None, None, None], labels, resolution

**`PatchDataset`** — Index-deterministic patch stream; resume by optimizer update, no RNG drift.

In [ ]:
class PatchDataset(Dataset):
    """Index-deterministic patch stream; resume by optimizer update, no RNG drift."""
    def __init__(self, store, subjects, patch, seed, length, corruption_probability=0.5, start=0):
        self.store, self.subjects, self.patch = store, list(subjects), tuple(patch)
        self.seed, self.length, self.corruption_probability, self.start = seed, length, corruption_probability, start

    def __len__(self):
        return self.length

    def __getitem__(self, index):
        rng = np.random.default_rng(np.random.SeedSequence([self.seed, self.start + index]))
        subject = self.subjects[int(rng.integers(len(self.subjects)))]
        images, labels, _ = self.store.load(subject)
        available = sample_availability(rng)
        images, labels = extract_patch(images, labels, available, self.patch, rng)
        images, labels, resolution = training_augmentation(images, labels, available, rng, self.corruption_probability)
        return dict(images=torch.from_numpy(images), labels=torch.from_numpy(labels.astype(np.int64)),
                    available=torch.from_numpy(available), resolution=torch.from_numpy(resolution), subject=subject)

In [ ]:
def make_loader(dataset):
    return DataLoader(dataset, batch_size=1, shuffle=False, num_workers=0,
                      pin_memory=torch.cuda.is_available())

**`SyntheticStore`** — Deterministic tiny blobs solely for execution tests, never a BraTS substitute.

In [ ]:
class SyntheticStore:
    """Deterministic tiny blobs solely for execution tests, never a BraTS substitute."""
    def __init__(self, count=12, size=24):
        self.records = {f"synthetic_{i:03}": {} for i in range(count)}
        self.size = size

    def load(self, subject):
        rng = np.random.default_rng(int(subject.split("_")[-1]))
        grid = np.indices((self.size,) * 3)
        distance = sum((g - self.size / 2) ** 2 for g in grid)
        brain = distance < (self.size * 0.44) ** 2
        labels = np.zeros((self.size,) * 3, np.uint8)
        labels[distance < (self.size * 0.24) ** 2] = 2
        labels[distance < (self.size * 0.16) ** 2] = 1
        labels[(grid[0] > self.size / 2) & (distance < (self.size * 0.16) ** 2)] = 3
        images = np.stack([(rng.normal(2, .2, labels.shape) + labels * (m + 1) * .2) * brain for m in range(4)]).astype(np.float32)
        metadata = dict(id=subject, spacing=[1., 1., 1.], canonical_shape=list(labels.shape),
                        canonical_affine=np.eye(4).tolist(), original_shape=list(labels.shape),
                        original_affine=np.eye(4).tolist(), orientation="RAS", synthetic=True)
        return normalize_foreground(images), labels, metadata

### Small visualization helpers
These figures use a prespecified middle slice; they do not select the most flattering prediction.

**`plot_subject`** — Middle superior-axis slice chosen without finding a favourable lesion.

In [ ]:
def plot_subject(images, labels, path, title="MRI and released segmentation"):
    """Middle superior-axis slice chosen without finding a favourable lesion."""
    index = images.shape[-1] // 2
    fig, axes = plt.subplots(1, 5, figsize=(15, 3))
    for i, name in enumerate(MODALITIES):
        axes[i].imshow(images[i, :, :, index].T, cmap="gray", origin="lower")
        axes[i].set_title(name.upper())
    axes[4].imshow(labels[:, :, index].T, cmap="tab10", vmin=0, vmax=3, origin="lower")
    axes[4].set_title("Internal classes 0–3")
    for ax in axes:
        ax.set_axis_off()
    fig.suptitle(title); fig.tight_layout(); fig.savefig(path, dpi=150); plt.close(fig)

**`plot_corrections`** — The same signed colour scale shows actual logit changes, not gate magnitude.

In [ ]:
def plot_corrections(images, truth, output, path, modality=3):
    """The same signed colour scale shows actual logit changes, not gate magnitude."""
    index = images.shape[-1] // 2
    base = output["base"].argmax(0).numpy()
    final = output["logits"].argmax(0).numpy()
    removed = (output["logits"] - output["contributions"][modality]).argmax(0).numpy()
    contribution = output["contributions"][modality, 3].numpy()
    limit = max(float(np.quantile(np.abs(contribution), .99)), 1e-5)
    panels = [(images[modality], "MRI", "gray", None), (truth, "Reference", "tab10", 3),
              (base, "Base", "tab10", 3), (final, "Final", "tab10", 3),
              (contribution, "ET logit contribution", "coolwarm", limit),
              (removed, "This detail removed", "tab10", 3)]
    fig, axes = plt.subplots(1, 6, figsize=(18, 3))
    for ax, (array, name, cmap, maximum) in zip(axes, panels):
        kwargs = {} if maximum is None else dict(vmin=-maximum if cmap == "coolwarm" else 0, vmax=maximum)
        ax.imshow(array[:, :, index].T, cmap=cmap, origin="lower", **kwargs)
        ax.set_title(name); ax.set_axis_off()
    fig.tight_layout(); fig.savefig(path, dpi=150); plt.close(fig)

**`plot_qualitative_paper`** — First prespecified validation patient; slice centres use reference tumour only.

In [ ]:
def plot_qualitative_paper(images, truth, output, directory, subject, spacing):
    """First prespecified validation patient; slice centres use reference tumour only."""
    directory = Path(directory)
    directory.mkdir(parents=True, exist_ok=True)
    tumour = np.argwhere(truth > 0)
    centre = np.median(tumour, axis=0).astype(int) if len(tumour) else np.asarray(truth.shape)//2
    base, final = output['base'].argmax(0).numpy(), output['logits'].argmax(0).numpy()
    colours = ListedColormap(['#000000', '#CC79A7', '#E69F00', '#009E73'])
    norm = BoundaryNorm([-.5, .5, 1.5, 2.5, 3.5], 4)
    source = dict(subject=subject, partition='validation', selection='first validation subject in saved split',
                  slice_selection='median reference tumour coordinates; volume centre when no tumour',
                  ras_indices=centre.tolist(), spacing_mm=list(spacing), classes={'0':'background','1':'NCR/NET','2':'edema','3':'ET'},
                  interpretation='Held-out illustration, not a representative-cohort claim or clinical explanation')
    native = {}
    fig, axes = plt.subplots(3, 4, figsize=(11, 9), layout='constrained')
    for row, (axis, label, xy) in enumerate(((2,'Axial',(0,1)), (1,'Coronal',(0,2)), (0,'Sagittal',(1,2)))):
        extent = (0, truth.shape[xy[0]]*spacing[xy[0]], 0, truth.shape[xy[1]]*spacing[xy[1]])
        for col, (array, title) in enumerate(((images[3], 'FLAIR'), (truth,'Reference'), (base,'Base'), (final,'Final'))):
            plane = np.take(array, centre[axis], axis=axis).T
            native[label+'_'+title] = plane
            kwargs = dict(cmap='gray') if col == 0 else dict(cmap=colours, norm=norm)
            im = axes[row,col].imshow(plane, origin='lower', extent=extent, interpolation='nearest', **kwargs)
            axes[row,col].set(title=title if row == 0 else '', xlabel='Distance (mm)')
            if col == 0:
                axes[row,col].set_ylabel(label+'; distance (mm)')
    bar = fig.colorbar(im, ax=axes[:,1:].ravel().tolist(), ticks=[0,1,2,3], shrink=.6)
    bar.ax.set_yticklabels(['background','NCR/NET','edema','ET'])
    artifacts = {'segmentation_views': save_paper_figure(fig, directory, subject+'_segmentation_views',
        'Three orthogonal views of the prespecified validation subject. Reference, base and final use identical class colours. Distances use voxel spacing; orientation is canonical RAS. Reference masks only select display slices, never inference features.', source)}
    index = centre[2]
    extent = (0, truth.shape[0]*spacing[0], 0, truth.shape[1]*spacing[1])
    for key, name, caption in (
        ('gates', 'spatial_gates', 'Window-averaged gate scores, not calibrated quality probabilities.'),
        ('contributions', 'et_contribution_maps', 'Actual additive ET logit contributions; positive adds ET evidence and negative subtracts it. Magnitude does not establish benefit.')):
        maps = output[key][:, 0 if key == 'gates' else 3, :, :, index].numpy()
        native[name] = maps
        bound = max(float(np.abs(maps).max()), 1e-8)
        fig, axes = plt.subplots(1, 4, figsize=(12, 3.8), layout='constrained')
        for ax, modality, plane in zip(axes, MODALITIES, maps):
            im = ax.imshow(plane.T, origin='lower', interpolation='nearest', extent=extent,
                           cmap='viridis' if key == 'gates' else 'RdBu_r',
                           vmin=0 if key == 'gates' else -bound, vmax=1 if key == 'gates' else bound)
            ax.set(title=modality.upper(), xlabel='R distance (mm)', ylabel='A distance (mm)')
        fig.colorbar(im, ax=axes, label='Gate score' if key == 'gates' else 'ET logit change', shrink=.7)
        artifacts[name] = save_paper_figure(fig, directory, subject+'_'+name,
            caption+' Same axial slice for all modalities; shared colour limits without percentile clipping.', source)
    np.savez_compressed(directory/(subject+'_display_source.npz'), **native)
    return artifacts

In [ ]:
DATASET_ROOT = None           # None discovers compatible mounted inputs
DATASET_VERSION = None        # fill with the version shown in Kaggle Input
DATASET_LICENSE_NOTE = "Mirror licence not independently verified; consult dataset card and original BraTS research/citation terms."
if not cfg.synthetic:
    cfg = replace(cfg, data_root=str(discover_dataset_root(DATASET_ROOT)), dataset_version=DATASET_VERSION)
    print("Verified existing mount:", cfg.data_root)

In [ ]:
if cfg.synthetic:
    store = SyntheticStore()
    subjects = [dict(id=s, patient_id=s) for s in store.records]
    discovery = dict(eligible=len(subjects), synthetic=True, excluded=[])
else:
    subjects, discovery = discover_subjects(cfg.data_root)
write_json(output_dir / "dataset_discovery.json", discovery)
print("Usable labelled subjects:", len(subjects), "Excluded:", len(discovery["excluded"]))

In [ ]:
if cfg.synthetic:
    integrity = dict(status="synthetic_only", subjects=len(subjects))
else:
    # Reads each labelled case once; can take several minutes. Progress is
    # saved after every subject, including failures.
    integrity = inventory_geometry(subjects, output_dir / "dataset_integrity.json")
    # Temporary preprocessing only: outside committed /kaggle/working outputs.
    store = SubjectStore(subjects, Path("/tmp/detailtrust_cache"), cfg.cache_gb)
write_json(output_dir / "dataset_integrity.json", integrity)

In [ ]:
split = make_split(subjects, cfg.split_seed)
existing_manifest = output_dir / "split_manifest.json"
if existing_manifest.exists():
    assert json.loads(existing_manifest.read_text())["sha256"] == split["sha256"], "Dataset/split changed during resume"
assert_split(split)
write_json(existing_manifest, split)
print({k: len(split[k]) for k in ("train", "validation", "test")}, "CUSTOM split")

In [ ]:
train_ids = split["train"][:min(cfg.train_subjects, len(split["train"]))]
val_ids = split["validation"][:cfg.val_subjects]
# Define the small-ET stratum using training labels only, before evaluation.
et_volumes = []
if not cfg.synthetic:
    # The integrity audit already measured these volumes. Avoid reading
    # and normalizing the entire training pool a second time.
    et_volumes = [row["et_volume_mm3"] for row in integrity
                  if row["id"] in train_ids and row["et_volume_mm3"] > 0]
else:
    for subject_id in train_ids:
        _, training_labels, geometry = store.load(subject_id)
        volume = float((training_labels == 3).sum() * np.prod(geometry["spacing"]))
        if volume > 0:
            et_volumes.append(volume)
cfg = replace(cfg, small_et_threshold_mm3=float(np.quantile(et_volumes, .25)) if et_volumes else None)
example_id = train_ids[0]
example_images, example_labels, example_geometry = store.load(example_id)
print("Example:", example_id, example_images.shape, np.unique(example_labels), example_geometry["spacing"])
plot_subject(example_images, example_labels, output_dir / "figures" / "preprocessing.png", "Normalized training example; labels never enter input normalization")

In [ ]:
dataset_evidence = dict(identifier=DATASET_SLUG if not cfg.synthetic else "synthetic_execution_fixture",
    source="https://www.kaggle.com/datasets/"+DATASET_SLUG, version=cfg.dataset_version,
    version_status="user_declared" if cfg.dataset_version else "not independently verified",
    licence=DATASET_LICENSE_NOTE, original_terms="https://www.med.upenn.edu/cbica/brats2020/data.html",
    discovery=discovery, official_test_labels_used=False, mounted_files_verified=not cfg.synthetic,
    patient_identity_limit="one BraTS20 subject ID per patient; no cross-release pooling",
    inventory_hash=file_hash(output_dir / "dataset_integrity.json"))
write_json(output_dir / "dataset_evidence.json", dataset_evidence)
write_json(output_dir / "experiment_config.json", asdict(cfg))

In [ ]:
patch_dataset = PatchDataset(store, train_ids, cfg.patch, cfg.seed, 2)
loader = make_loader(patch_dataset)
example_batch = next(iter(loader))
assert example_batch["images"].shape == (1, 4, *(n+8 for n in cfg.patch))
assert example_batch["labels"].shape == (1, *cfg.patch)
assert set(example_batch["labels"].unique().tolist()) <= {0, 1, 2, 3}
print({k: tuple(v.shape) for k, v in example_batch.items() if isinstance(v, torch.Tensor)})

## C/D — Baseline and DetailTrust-Net
    The raw baseline uses four MRI channels plus four availability channels. The
    same five-level residual U-Net also supplies the proposed model's base logits
    and 32-channel features. The model has **16,934,644 parameters** in B5.

    `L_m = Σ softmax(α_m)_s G_σs(X_m)` with σ = 0.5/0.8/1.2 **mm**; `H=X-L`.
    `r_m=R(F,H_m,e_m)-R(F,0,e_m)` uses 32+1+4=37 input channels.
    The gate receives 32+1+3+4+4=44 channels. Final foreground logits are
    `z=z0+Σ a*q*r`, and the base background logit is unchanged. There is no later
    convolution. A gate is a learned multiplier, not a calibrated quality probability.

**`availability_maps`** — images [B,4,D,H,W]; available [B,4]. Reject absent evidence.

In [ ]:
def availability_maps(images, available):
    """images [B,4,D,H,W]; available [B,4]. Reject absent evidence."""
    assert images.ndim == 5 and images.shape[1] == 4
    assert available.shape == images.shape[:2]
    assert torch.all((available == 0) | (available == 1))
    assert torch.all(available.sum(1) > 0), "At least one MRI must be present."
    maps = available.to(images).view(*available.shape, 1, 1, 1)
    return images * maps, maps.expand_as(images)

**`gaussian_kernel`** — Convert physical sigma to voxels; all taps sum to one.

In [ ]:
def gaussian_kernel(sigma_mm: float, spacing_mm: float, radius: int, device, dtype):
    """Convert physical sigma to voxels; all taps sum to one."""
    assert sigma_mm > 0 and spacing_mm > 0
    coordinates = torch.arange(-radius, radius + 1, device=device, dtype=dtype)
    kernel = torch.exp(-0.5 * (coordinates / (sigma_mm / spacing_mm)) ** 2)
    return kernel / kernel.sum()

**`gaussian_blur`** — Separable, differentiable, nine-tap filtering on 1 mm data.

In [ ]:
def gaussian_blur(images, sigma_mm, spacing=(1., 1., 1.)):
    """Separable, differentiable, nine-tap filtering on 1 mm data."""
    result = images
    channels = images.shape[1]
    for axis, step in enumerate(spacing):
        radius = max(4, math.ceil(3 * 1.2 / step))
        kernel = gaussian_kernel(sigma_mm, step, radius, images.device, images.dtype)
        shape = [1, 1, 1]
        shape[axis] = kernel.numel()
        weight = kernel.view(1, 1, *shape).expand(channels, 1, *shape)
        padding = [0] * 6
        padding[2 * (2 - axis):2 * (2 - axis) + 2] = [radius, radius]
        assert result.shape[axis + 2] > radius, "Patch is too small for reflection."
        result = F.conv3d(F.pad(result, padding, mode="reflect"), weight, groups=channels)
    return result

**`GaussianSplit`** — Twelve global logits; no subject-dependent smoothing network.

In [ ]:
class GaussianSplit(nn.Module):
    """Twelve global logits; no subject-dependent smoothing network."""
    def __init__(self, learned=True):
        super().__init__()
        if learned:
            self.alpha = nn.Parameter(torch.tensor([[-1., 1., -1.]]).repeat(4, 1))
        else:
            self.register_parameter("alpha", None)

    def forward(self, images, spacing=(1., 1., 1.)):
        if self.alpha is None:
            low = gaussian_blur(images, 0.8, spacing)
        else:
            weights = self.alpha.softmax(-1)
            low = torch.zeros_like(images)
            for index, sigma in enumerate((0.5, 0.8, 1.2)):
                low = low + gaussian_blur(images, sigma, spacing) * weights[:, index].view(1, 4, 1, 1, 1)
        return low, images - low

**`HaarSplit`** — One-level orthonormal Haar: zero high bands, inverse low band, H=X-L.

Pairwise averaging followed by repeat is exactly that low reconstruction.
Even dimensions and an even (four-voxel) halo preserve the pair origin.

In [ ]:
class HaarSplit(nn.Module):
    """One-level orthonormal Haar: zero high bands, inverse low band, H=X-L.

    Pairwise averaging followed by repeat is exactly that low reconstruction.
    Even dimensions and an even (four-voxel) halo preserve the pair origin.
    """
    def forward(self, images, spacing=(1., 1., 1.)):
        assert all(n % 2 == 0 for n in images.shape[-3:])
        low = F.avg_pool3d(images, 2)
        for axis in (2, 3, 4):
            low = low.repeat_interleave(2, dim=axis)
        return low, images - low

**`ResidualBlock`** — Two bias-free convolutions, eight-group affine norms, projected skip.

In [ ]:
class ResidualBlock(nn.Module):
    """Two bias-free convolutions, eight-group affine norms, projected skip."""
    def __init__(self, in_channels, out_channels):
        super().__init__()
        self.conv1 = nn.Conv3d(in_channels, out_channels, 3, padding=1, bias=False)
        self.norm1 = nn.GroupNorm(8, out_channels)
        self.conv2 = nn.Conv3d(out_channels, out_channels, 3, padding=1, bias=False)
        self.norm2 = nn.GroupNorm(8, out_channels)
        self.projection = nn.Conv3d(in_channels, out_channels, 1, bias=False)

    def forward(self, images):
        features = F.leaky_relu(self.norm1(self.conv1(images)), 0.01)
        features = self.norm2(self.conv2(features)) + self.projection(images)
        return F.leaky_relu(features, 0.01)

**`run_block`** — Non-reentrant checkpointing changes memory use, not the model equation.

In [ ]:
def run_block(block, features, recompute=False):
    """Non-reentrant checkpointing changes memory use, not the model equation."""
    if recompute and torch.is_grad_enabled():
        return checkpoint(block, features, use_reentrant=False)
    return block(features)

**`ResidualUNet`** — 8-channel input -> four logits and a 32-channel full-resolution feature.

In [ ]:
class ResidualUNet(nn.Module):
    """8-channel input -> four logits and a 32-channel full-resolution feature."""
    def __init__(self, checkpoint_blocks=True):
        super().__init__()
        self.checkpoint_blocks = checkpoint_blocks
        self.encoders = nn.ModuleList(ResidualBlock(a, b) for a, b in
                                     zip((8,) + WIDTHS[:-1], WIDTHS))
        self.decoders = nn.ModuleList(ResidualBlock(a + b, b) for a, b in
                                     zip(WIDTHS[:0:-1], WIDTHS[-2::-1]))
        self.head = nn.Conv3d(32, 4, 1, bias=True)

    def forward(self, images):
        assert all(n % 16 == 0 for n in images.shape[-3:])
        skips = []
        features = images
        for index, block in enumerate(self.encoders):
            if index:
                features = F.avg_pool3d(features, 2)
            features = run_block(block, features, self.checkpoint_blocks)
            skips.append(features)
        for block, skip in zip(self.decoders, reversed(skips[:-1])):
            features = F.interpolate(features, size=skip.shape[-3:], mode="trilinear", align_corners=False)
            features = run_block(block, torch.cat((features, skip), 1), self.checkpoint_blocks)
        return self.head(features), features

**`modality_identity`** — A four-channel one-hot map, not a learned embedding.

In [ ]:
def modality_identity(reference, modality):
    """A four-channel one-hot map, not a learned embedding."""
    identity = reference.new_zeros((reference.shape[0], 4, 1, 1, 1))
    identity[:, modality] = 1
    return identity.expand(-1, -1, *reference.shape[-3:])

**`DetailCorrection`** — Shared R: 37 -> 16 -> 16 -> 3. No normalization or dropout.

In [ ]:
class DetailCorrection(nn.Module):
    """Shared R: 37 -> 16 -> 16 -> 3. No normalization or dropout."""
    def __init__(self):
        super().__init__()
        self.layers = nn.Sequential(nn.Conv3d(37, 16, 3, padding=1), nn.ReLU(),
                                    nn.Conv3d(16, 16, 3, padding=1), nn.ReLU(),
                                    nn.Conv3d(16, 3, 1))
        nn.init.normal_(self.layers[-1].weight, std=0.001)
        nn.init.zeros_(self.layers[-1].bias)

    def forward(self, features, detail, identity, anchored=True):
        correction = self.layers(torch.cat((features, detail, identity), 1))
        if anchored:
            correction = correction - self.layers(torch.cat((features, torch.zeros_like(detail), identity), 1))
        return correction

**`SpatialUtilityGate`** — 32+1+3+4+4=44 channels -> one independent sigmoid at each voxel.

In [ ]:
class SpatialUtilityGate(nn.Module):
    """32+1+3+4+4=44 channels -> one independent sigmoid at each voxel."""
    def __init__(self):
        super().__init__()
        self.layers = nn.Sequential(nn.Conv3d(44, 16, 1), nn.ReLU(),
                                    nn.Conv3d(16, 16, 3, padding=1, groups=16), nn.ReLU(),
                                    nn.Conv3d(16, 1, 1), nn.Sigmoid())

    def forward(self, features, detail, correction, identity, flags):
        inputs = torch.cat((features, detail, correction, identity, flags), 1)
        assert inputs.shape[1] == 44
        return self.layers(inputs)

**`fuse_logits`** — r [B,M,3,D,H,W], q [B,M,1,D,H,W]; return exact foreground sum.

In [ ]:
def fuse_logits(base_logits, corrections, gates, available, retention=None):
    """r [B,M,3,D,H,W], q [B,M,1,D,H,W]; return exact foreground sum."""
    weight = available[:, :, None, None, None, None].to(corrections)
    if retention is not None:
        weight = weight * retention[:, :, None, None, None, None]
    foreground = weight * gates * corrections
    background = torch.zeros_like(foreground[:, :, :1])
    contributions = torch.cat((background, foreground), 2).float()
    logits = base_logits.float() + contributions.sum(1)
    return logits, contributions

**`DetailTrustNet`** — arch.md reference model. Ablations are explicit constructor arguments.

In [ ]:
class DetailTrustNet(nn.Module):
    """arch.md reference model. Ablations are explicit constructor arguments."""
    def __init__(self, split="learned", anchored=True, gate="spatial", checkpoint_blocks=True):
        super().__init__()
        self.architecture_config = dict(split=split, anchored=anchored, gate=gate,
                                        checkpoint_blocks=checkpoint_blocks)
        self.split_kind, self.anchored, self.gate_kind = split, anchored, gate
        self.backbone = ResidualUNet(checkpoint_blocks)
        self.splitter = GaussianSplit(split == "learned") if split in ("learned", "fixed") else HaarSplit() if split == "haar" else None
        self.refiner = DetailCorrection() if gate != "raw" else None
        self.gate = SpatialUtilityGate() if gate == "spatial" else None
        self.global_logits = nn.Parameter(torch.zeros(4)) if gate == "global" else None
        self.freeze_candidates = False

    def candidates(self, images, available, halo=4, compute_detail=True):
        """Filter the halo first; feed only the central patch to the U-Net."""
        images, flags = availability_maps(images, available)
        low, detail = self.splitter(images) if self.splitter else (images, images)
        crop = (slice(None), slice(None)) + (slice(halo, -halo),) * 3 if halo else (...,)
        low, detail, flags = low[crop], detail[crop], flags[crop]
        base, features = self.backbone(torch.cat((low, flags), 1))
        corrections = []
        for modality in range(4):
            if not compute_detail or self.refiner is None or not bool(available[:, modality].any()):
                correction = torch.zeros_like(base[:, 1:])
            else:
                identity = modality_identity(features, modality)
                def refine(f, h, e):
                    return self.refiner(f, h, e, self.anchored)
                args = (features, detail[:, modality:modality + 1], identity)
                correction = checkpoint(refine, *args, use_reentrant=False) if self.backbone.checkpoint_blocks and torch.is_grad_enabled() else refine(*args)
                correction = correction * available[:, modality, None, None, None, None]
            corrections.append(correction)
        return base, features, detail, flags, torch.stack(corrections, 1)

    def forward(self, images, available, stage="C", halo=4, retention=None):
        with torch.no_grad() if self.freeze_candidates else nullcontext():
            base, features, detail, flags, corrections = self.candidates(images, available, halo, stage != "A")
        gates = []
        for modality in range(4):
            if stage in ("A", "B") or self.gate_kind in ("ones", "raw"):
                gate = torch.ones_like(base[:, :1])
            elif self.gate_kind == "global":
                gate = self.global_logits[modality].sigmoid().expand_as(base[:, :1])
            else:
                gate = self.gate(features, detail[:, modality:modality + 1], corrections[:, modality],
                                 modality_identity(features, modality), flags)
            gates.append(gate)
        gates = torch.stack(gates, 1)
        if stage == "A":
            corrections = corrections * 0
        logits, contributions = fuse_logits(base, corrections, gates, available, retention)
        return dict(logits=logits, base=base.float(), corrections=corrections.float(),
                    gates=gates.float(), contributions=contributions,
                    detail=detail, available=available)

**`build_model`** — Raw baselines have no dormant splitter/refiner/gate parameters.

In [ ]:
def build_model(variant, checkpoint_blocks=True):
    """Raw baselines have no dormant splitter/refiner/gate parameters."""
    raw = variant["gate"] == "raw"
    return DetailTrustNet(split="raw" if raw else variant["split"],
                          anchored=variant["anchored"], gate=variant["gate"],
                          checkpoint_blocks=checkpoint_blocks)

In [ ]:
def parameter_counts(model):
    return dict(total=sum(p.numel() for p in model.parameters()),
                trainable=sum(p.numel() for p in model.parameters() if p.requires_grad),
                by_child={n: sum(p.numel() for p in m.parameters()) for n, m in model.named_children()})

In [ ]:
reference_model = DetailTrustNet(checkpoint_blocks=cfg.checkpoint_blocks).to(device)
print(parameter_counts(reference_model))
assert parameter_counts(reference_model)["total"] == 16_934_644
with torch.no_grad():
    small_x = torch.randn(1, 4, 24, 24, 24, device=device)
    small_a = torch.tensor([[1., 1., 1., 1.]], device=device)
    small_result = reference_model(small_x, small_a)
print({k: tuple(v.shape) for k, v in small_result.items()})
torch.testing.assert_close(small_result["logits"][:, 0], small_result["base"][:, 0])
del reference_model, small_result, small_x
if device.type == "cuda": torch.cuda.empty_cache()

## E — Training objectives
    A: train the base and smoothing with CE plus WT/TC/ET soft-Dice.
    B: gates equal one; independently drop detail branches with probability .25;
    use final segmentation + .3 base segmentation + .1 available-branch auxiliary mean.
    C: freeze the candidates and train gates. In the all-candidates-present context,
    `delta=CE(z_all-r_m,Y)-CE(z_all,Y)`. Positive means helpful. Average over 3³,
    divide by .1, apply sigmoid, and **detach**. Gate BCE gives equal weight to WT
    and background when both exist. No training label enters an inference feature.

In [ ]:
def region_probabilities(logits):
    probabilities = logits.float().softmax(1)
    return torch.stack((probabilities[:, 1:].sum(1), probabilities[:, 1] + probabilities[:, 3], probabilities[:, 3]), 1)

**`region_labels`** — WT, TC, ET in that order, derived from exclusive internal labels.

In [ ]:
def region_labels(labels):
    """WT, TC, ET in that order, derived from exclusive internal labels."""
    return torch.stack((labels > 0, (labels == 1) | (labels == 3), labels == 3), 1)

**`segmentation_loss`** — CE + mean region soft-Dice, reduced per patient then across patients.

In [ ]:
def segmentation_loss(logits, labels):
    """CE + mean region soft-Dice, reduced per patient then across patients."""
    logits = logits.float()
    truth, probabilities = region_labels(labels).float(), region_probabilities(logits)
    axes = (2, 3, 4)
    intersection = (truth * probabilities).sum(axes)
    dice = (2 * intersection + 1e-5) / (truth.sum(axes) + probabilities.sum(axes) + 1e-5)
    return F.cross_entropy(logits, labels) + (1 - dice).mean()

**`utility_targets`** — t_m = sigmoid(pool(CE(z_all-r_m)-CE(z_all))/tau), completely detached.

Sum all AVAILABLE ungated candidate corrections. Positive delta means that
removing this correction increases CE, so the correction helped. Pool only
spatial axes, using a truncated neighbourhood at volume boundaries.

In [ ]:
def utility_targets(base, corrections, available, labels, tau=0.1):
    """t_m = sigmoid(pool(CE(z_all-r_m)-CE(z_all))/tau), completely detached.

    Sum all AVAILABLE ungated candidate corrections. Positive delta means that
    removing this correction increases CE, so the correction helped. Pool only
    spatial axes, using a truncated neighbourhood at volume boundaries.
    """
    with torch.no_grad():
        ones = torch.ones_like(corrections[:, :, :1])
        all_logits, candidate_contributions = fuse_logits(base, corrections, ones, available)
        reference_ce = F.cross_entropy(all_logits.float(), labels, reduction="none")
        targets, deltas = [], []
        for modality in range(4):
            without = all_logits - candidate_contributions[:, modality]
            delta = F.cross_entropy(without, labels, reduction="none") - reference_ce
            smooth = F.avg_pool3d(delta[:, None], 3, stride=1, padding=1, count_include_pad=False)
            targets.append(torch.sigmoid(smooth / tau))
            deltas.append(delta[:, None])
        return torch.stack(targets, 1).detach(), torch.stack(deltas, 1).detach()

**`balanced_gate_bce`** — Equal WT/background weighting per subject and available modality.

In [ ]:
def balanced_gate_bce(gates, targets, labels, available):
    """Equal WT/background weighting per subject and available modality."""
    losses = F.binary_cross_entropy(gates.float().clamp(1e-7, 1 - 1e-7), targets.float(), reduction="none")
    terms = []
    for patient in range(labels.shape[0]):
        masks = [labels[patient] > 0, labels[patient] == 0]
        masks = [mask for mask in masks if bool(mask.any())]
        for modality in range(4):
            if bool(available[patient, modality]):
                terms.append(torch.stack([losses[patient, modality, 0][mask].mean() for mask in masks]).mean())
    return torch.stack(terms).mean()

**`resolution_occupancy_loss`** — Positive row-normalized adaptive averages for a declared coarse lattice.

In [ ]:
def resolution_occupancy_loss(logits, labels, coarse_shape):
    """Positive row-normalized adaptive averages for a declared coarse lattice."""
    probabilities = logits.float().softmax(1)
    one_hot = F.one_hot(labels, 4).permute(0, 4, 1, 2, 3).float()
    truth = F.adaptive_avg_pool3d(one_hot, coarse_shape)
    prediction = F.adaptive_avg_pool3d(probabilities, coarse_shape)
    return -(truth * prediction.clamp_min(1e-7).log()).sum(1).mean()

**`stage_objective`** — Return the scalar loss and separately logged, named components.

In [ ]:
def stage_objective(outputs, labels, stage, cfg, variant, resolution=None):
    """Return the scalar loss and separately logged, named components."""
    available, base, corrections = outputs["available"], outputs["base"], outputs["corrections"]
    terms = {}
    if stage in ("A", "RAW"):
        terms["segmentation"] = segmentation_loss(outputs["logits"], labels)
        loss = terms["segmentation"]
    elif stage == "B":
        retention = (torch.rand_like(available) >= cfg.branch_dropout).float()
        logits, _ = fuse_logits(base, corrections, torch.ones_like(outputs["gates"]), available, retention)
        terms["segmentation"] = segmentation_loss(logits, labels)
        terms["base"] = segmentation_loss(base, labels)
        auxiliary = []
        for patient in range(labels.shape[0]):
            for modality in range(4):
                if bool(available[patient, modality]):
                    addition = F.pad(corrections[patient:patient+1, modality], (0, 0, 0, 0, 0, 0, 1, 0))
                    auxiliary.append(segmentation_loss(base[patient:patient+1] + addition, labels[patient:patient+1]))
        terms["branch_auxiliary"] = torch.stack(auxiliary).mean()
        loss = terms["segmentation"] + 0.3 * terms["base"] + 0.1 * terms["branch_auxiliary"]
    elif stage == "C":
        terms["segmentation"] = segmentation_loss(outputs["logits"], labels)
        loss = terms["segmentation"]
        if variant["utility"]:
            target, delta = utility_targets(base, corrections, available, labels, cfg.utility_tau)
            if variant["shuffle"]:
                # Same marginal targets; destroy spatial alignment within each branch.
                flat = target.flatten(3)
                order = torch.randperm(flat.shape[-1], device=flat.device)
                target = flat[..., order].reshape_as(target)
            terms["utility_bce"] = balanced_gate_bce(outputs["gates"], target, labels, available)
            terms["utility_target_mean"] = target.mean()
            loss = loss + cfg.utility_weight * terms["utility_bce"]
    else:
        raise ValueError(stage)
    if variant["resolution_loss"] and resolution is not None and bool((resolution > 0).all()):
        shape = tuple(int(x) for x in resolution[0].tolist())
        terms["resolution_occupancy"] = resolution_occupancy_loss(outputs["logits"], labels, shape)
        loss = loss + 0.1 * terms["resolution_occupancy"]
    terms["total"] = loss
    return loss, {name: float(value.detach()) for name, value in terms.items()}

## F — Metrics and conditional stability
    Report patient macro averages, preserve empty cases, and use physical millimetres
    for surfaces. The stability bound considers attenuation of already-computed
    contributions only. It does not certify correctness, input corruption, or
    whole-modality removal. One-empty HD95 is undefined (`null`), not zero.

**`binary_metrics`** — Macro aggregation is performed later; all empty-case decisions are explicit.

In [ ]:
def binary_metrics(prediction, truth, spacing, tolerance_mm=1.0):
    """Macro aggregation is performed later; all empty-case decisions are explicit."""
    prediction, truth = prediction.astype(bool), truth.astype(bool)
    tp = int((prediction & truth).sum())
    fp = int((prediction & ~truth).sum())
    fn = int((~prediction & truth).sum())
    tn = int((~prediction & ~truth).sum())
    p, t = tp + fp, tp + fn
    both_empty, one_empty = p == 0 and t == 0, (p == 0) != (t == 0)
    result = dict(dice=2 * tp / (p + t) if p + t else 1.,
                  iou=tp / (tp + fp + fn) if tp + fp + fn else 1.,
                  precision=tp / p if p else None, recall=tp / t if t else None,
                  specificity=tn / (tn + fp) if tn + fp else None,
                  tp=tp, fp=fp, fn=fn, tn=tn, truth_voxels=t, prediction_voxels=p,
                  both_empty=both_empty, one_empty=one_empty,
                  fp_mm3=float(fp * np.prod(spacing)),
                  absolute_volume_error_mm3=float(abs(p - t) * np.prod(spacing)),
                  undefined_policy="precision null if prediction empty; recall null if truth empty; HD95 null if exactly one empty")
    if both_empty or one_empty:
        result.update(hd95_mm=0. if both_empty else None,
                      surface_dice=1. if both_empty else 0.)
        return result
    structure = ndimage.generate_binary_structure(3, 1)
    ps = prediction & ~ndimage.binary_erosion(prediction, structure=structure, border_value=0)
    ts = truth & ~ndimage.binary_erosion(truth, structure=structure, border_value=0)
    p_to_t = ndimage.distance_transform_edt(~ts, sampling=spacing)[ps]
    t_to_p = ndimage.distance_transform_edt(~ps, sampling=spacing)[ts]
    result["hd95_mm"] = float(np.percentile(np.concatenate((p_to_t, t_to_p)), 95))
    result["surface_dice"] = float(((p_to_t <= tolerance_mm).sum() + (t_to_p <= tolerance_mm).sum()) / (len(p_to_t) + len(t_to_p)))
    return result

**`lesion_metrics`** — 26-connected ET lesions; one-to-one maximum-cardinality overlap matching.

In [ ]:
def lesion_metrics(prediction, truth):
    """26-connected ET lesions; one-to-one maximum-cardinality overlap matching."""
    from scipy.sparse import csr_matrix
    from scipy.sparse.csgraph import maximum_bipartite_matching
    pred, npred = ndimage.label(prediction, structure=np.ones((3, 3, 3)))
    true, ntrue = ndimage.label(truth, structure=np.ones((3, 3, 3)))
    overlap = (pred > 0) & (true > 0)
    pairs = np.unique(np.stack((pred[overlap], true[overlap]), 1), axis=0)
    if len(pairs):
        graph = csr_matrix((np.ones(len(pairs)), (pairs[:, 0]-1, pairs[:, 1]-1)), shape=(npred, ntrue))
        matched = int((maximum_bipartite_matching(graph, perm_type="column") >= 0).sum())
    else:
        matched = 0
    return dict(predicted_lesions=npred, reference_lesions=ntrue, matched_lesions=matched,
                lesion_precision=matched / npred if npred else None,
                lesion_recall=matched / ntrue if ntrue else None)

In [ ]:
def subject_metrics(prediction, truth, spacing, subject, experiment, condition, split, tolerance=1.):
    rows = []
    for region, classes in (REGIONS | CLASSES).items():
        row = binary_metrics(np.isin(prediction, classes), np.isin(truth, classes), spacing, tolerance)
        row.update(subject=subject, experiment=experiment, condition=condition, split=split, region=region,
                   metric_scope="region" if region in REGIONS else "exclusive_class",
                   aggregation="per_patient", surface_tolerance_mm=tolerance,
                   surface_convention="6-connected boundary voxel centres; unweighted surface point Dice",
                   hd95_convention="95th percentile of concatenated directed distances")
        if region == "ET":
            row.update(lesion_metrics(prediction == 3, truth == 3))
        rows.append(row)
    return rows

**`aggregate_metrics`** — Separate each experiment/condition/split/region; count undefined cases.

In [ ]:
def aggregate_metrics(rows):
    """Separate each experiment/condition/split/region; count undefined cases."""
    groups = {}
    for row in rows:
        key = tuple(row[k] for k in ("experiment", "condition", "split", "region"))
        groups.setdefault(key, []).append(row)
    results = []
    for key, values in groups.items():
        summary = dict(zip(("experiment", "condition", "split", "region"), key))
        summary.update(subjects=[v["subject"] for v in values], subject_count=len(values), aggregation="patient_macro")
        for metric in ("dice", "iou", "precision", "recall", "specificity", "hd95_mm", "surface_dice", "fp_mm3", "absolute_volume_error_mm3", "lesion_precision", "lesion_recall"):
            data = np.asarray([v[metric] for v in values if v.get(metric) is not None], float)
            summary[metric] = dict(mean=float(data.mean()) if len(data) else None,
                                   std=float(data.std(ddof=1)) if len(data) > 1 else None,
                                   median=float(np.median(data)) if len(data) else None,
                                   q05=float(np.quantile(data, .05)) if len(data) else None,
                                   q95=float(np.quantile(data, .95)) if len(data) else None,
                                   valid_count=len(data), undefined_count=len(values)-len(data))
        summary["both_empty_count"] = sum(v["both_empty"] for v in values)
        summary["one_empty_count"] = sum(v["one_empty"] for v in values)
        results.append(summary)
    return results

**`stability_bound`** — base [4,...], c [4 modalities,4 classes,...], fixed-context attenuation.

In [ ]:
def stability_bound(base, contributions, tolerance=1e-5):
    """base [4,...], c [4 modalities,4 classes,...], fixed-context attenuation."""
    logits = base + contributions.sum(0)
    winner = logits.argmax(0)
    winning_logit = torch.gather(logits, 0, winner[None])[0]
    winning_contributions = torch.gather(contributions, 1, winner[None, None].expand(contributions.shape[0], 1, *winner.shape))
    difference = winning_contributions - contributions
    bounds = winning_logit[None] - logits - difference.clamp_min(0).sum(0)
    bounds.scatter_(0, winner[None], float("inf"))
    minimum = bounds.min(0).values
    return minimum, minimum > tolerance

**`verify_stability_enumeration`** — Compare analytic rival margins against all 16 binary retention subsets.

In [ ]:
def verify_stability_enumeration(base, contributions, tolerance=1e-5):
    """Compare analytic rival margins against all 16 binary retention subsets."""
    logits = base + contributions.sum(0)
    winner = logits.argmax(0)
    worst = torch.full_like(logits, float("inf"))
    for bits in itertools.product((0., 1.), repeat=4):
        weights = contributions.new_tensor(bits).view(4, *([1] * (contributions.ndim - 1)))
        scores = base + (weights * contributions).sum(0)
        margins = torch.gather(scores, 0, winner[None]) - scores
        margins.scatter_(0, winner[None], float("inf"))
        worst = torch.minimum(worst, margins)
    bound, stable = stability_bound(base, contributions, tolerance)
    error = float((bound - worst.min(0).values).abs().max())
    return dict(max_margin_error=error, passed=error <= tolerance,
                subset_count=16, voxel_count=winner.numel())

**`paired_patient_comparison`** — Resample patients after averaging their matched region/condition scores.

In [ ]:
def paired_patient_comparison(rows, candidate, baseline, bootstrap_seed=42, partition="validation", endpoint="overall"):
    """Resample patients after averaging their matched region/condition scores."""
    scores = {}
    for row in rows:
        if row["split"] != partition or "_remove_detail_" in row["condition"] or row["condition"].endswith("_base_only"):
            continue
        if endpoint == "degraded" and row["condition"].startswith("clean"):
            continue
        if endpoint == "clean_ET" and (row["condition"] != "clean_m15" or row["region"] != "ET"):
            continue
        if row["experiment"] in (candidate, baseline) and row["region"] in REGIONS:
            key = (row["subject"], row["condition"], row["region"], row["split"])
            scores.setdefault(key, {})[row["experiment"]] = row["dice"]
    patient_values = {}
    for key, pair in scores.items():
        if candidate in pair and baseline in pair:
            patient_values.setdefault(key[0], []).append((pair[candidate], pair[baseline]))
    if not patient_values:
        return dict(status="not_run", reason="No matched patient-condition records")
    pairs = np.asarray([np.mean(values, axis=0) for values in patient_values.values()])
    difference = pairs[:, 0] - pairs[:, 1]
    rng = np.random.default_rng(bootstrap_seed)
    sampled = rng.choice(difference, size=(2000, len(difference)), replace=True).mean(1)
    base_mean = float(pairs[:, 1].mean())
    return dict(status="completed", candidate=candidate, baseline=baseline,
                partition=partition, endpoint=endpoint,
                patients=sorted(patient_values), patient_count=len(difference),
                absolute_dice_difference=float(difference.mean()),
                relative_difference=float(difference.mean() / base_mean) if base_mean else None,
                patient_bootstrap_ci95=np.quantile(sampled, [.025, .975]).tolist() if len(difference) > 1 else None,
                replicates=2000, unit="patient", warning="Conditional on trained weights; not training-seed uncertainty")

### Full-volume evaluation and explanation accounting
    Blend logits, base scores and contributions with identical window weights.
    Apply softmax/argmax only after blending. All 15 nonempty MRI subsets are
    distinct from the 16 detail-retention subsets used to check the bound.
    Degraded validation uses development corruptions; final testing adds held-out
    ghost/ringing families. All variants get identical deterministic conditions.

In [ ]:
def synchronized_time(device):
    if device.type == "cuda":
        torch.cuda.synchronize(device)
    return time.perf_counter()

In [ ]:
def autocast_context(device, precision):
    return torch.autocast(device_type=device.type, dtype=torch.float16,
                          enabled=device.type == "cuda" and precision == "amp_fp16")

In [ ]:
def window_starts(length, patch):
    stride = max(1, patch // 2)
    starts = list(range(0, max(1, length - patch + 1), stride))
    if starts[-1] != length - patch:
        starts.append(length - patch)
    return starts

**`sliding_window`** — Blend base and each c with the SAME constant overlap weights on CPU.

No softmax blending or nonlinear postprocessing. Filtering receives a real
four-voxel halo. Diagnostics q and r are window averages; q_bar*r_bar does
NOT equal c_bar and must never replace the accumulated contributions.

In [ ]:
@torch.no_grad()
def sliding_window(model, images, available, cfg, device, stage="C", analysis=False):
    """Blend base and each c with the SAME constant overlap weights on CPU.

    No softmax blending or nonlinear postprocessing. Filtering receives a real
    four-voxel halo. Diagnostics q and r are window averages; q_bar*r_bar does
    NOT equal c_bar and must never replace the accumulated contributions.
    """
    model.eval()
    slices = image_crop(images, available)
    crop = np.asarray(images[(slice(None),) + slices])
    original_crop_shape = crop.shape[1:]
    extra = [(0, max(0, p - n)) for n, p in zip(original_crop_shape, cfg.patch)]
    crop = np.pad(crop, [(0, 0)] + extra, mode="constant")
    padded = np.pad(crop, [(0, 0), (4, 4), (4, 4), (4, 4)], mode="reflect")
    shape = crop.shape[1:]
    accum = {"base": torch.zeros(4, *shape), "contributions": torch.zeros(4, 4, *shape)}
    if analysis:
        accum.update(gates=torch.zeros(4, 1, *shape), corrections=torch.zeros(4, 3, *shape),
                     detail_energy=torch.zeros(4, 1, *shape))
    count = torch.zeros(shape)
    flags = torch.as_tensor(available, device=device)[None]
    windows = 0
    started = synchronized_time(device)
    for origin in itertools.product(*(window_starts(n, p) for n, p in zip(shape, cfg.patch))):
        target = tuple(slice(s, s + p) for s, p in zip(origin, cfg.patch))
        source = tuple(slice(s, s + p + 8) for s, p in zip(origin, cfg.patch))
        patch = torch.from_numpy(padded[(slice(None),) + source].copy())[None].to(device)
        with autocast_context(device, cfg.precision):
            output = model(patch, flags, stage=stage)
        for name in accum:
            value = output["detail"][0, :, None].abs() if name == "detail_energy" else output[name][0]
            accum[name][(...,) + target] += value.float().cpu()
        count[target] += 1
        windows += 1
    elapsed = synchronized_time(device) - started
    assert bool((count > 0).all())
    result = {}
    unpad = tuple(slice(0, n) for n in original_crop_shape)
    for name, value in accum.items():
        value = (value / count)[(...,) + unpad]
        full = torch.zeros(*value.shape[:-3], *images.shape[1:])
        full[(...,) + slices] = value
        result[name] = full
    result["logits"] = result["base"] + result["contributions"].sum(0)
    result["runtime"] = dict(seconds=elapsed, windows=windows, patch=list(cfg.patch),
                             full_shape=list(images.shape[1:]), warmup=False)
    return result

**`evaluation_conditions`** — Freeze this suite before testing. Ghost/ringing are held out from training.

In [ ]:
def evaluation_conditions(cfg, final=False):
    """Freeze this suite before testing. Ghost/ringing are held out from training."""
    conditions = []
    # Measure the complete-input model first if the session cannot finish the suite.
    for bits in (15, *range(1, 15)):
        flags = [(bits >> i) & 1 for i in range(4)]
        conditions.append(dict(id=f"clean_m{bits:02}", available=flags, corruption=None))
    families = ("noise",) if cfg.robustness == "smoke" else (("ghost", "ringing") if final else ("noise", "resolution"))
    levels = (1,) if cfg.robustness == "smoke" else (1, 2)
    masks = (15,) if cfg.robustness == "smoke" else (15, 13)  # 13 has T1ce absent.
    for family, level, bits in itertools.product(families, levels, masks):
        flags = [(bits >> i) & 1 for i in range(4)]
        severity = {"noise": .075 * level, "resolution": 1.5 + level,
                    "ghost": .15 * level, "ringing": .2 * level}[family]
        conditions.append(dict(id=f"{family}{level}_m{bits:02}_flair", available=flags,
                               corruption=dict(family=family, severity=severity, modality=3, local=True, axis=2)))
    return conditions

**`benchmark_inference`** — One whole-volume warmup followed by repeated timed reconstructions.

In [ ]:
def benchmark_inference(model, store, subject, cfg, device):
    """One whole-volume warmup followed by repeated timed reconstructions."""
    images, _, _ = store.load(subject)
    available = np.ones(4, np.float32)
    warmup = sliding_window(model, images, available, cfg, device)
    del warmup
    if device.type == "cuda":
        torch.cuda.reset_peak_memory_stats(device)
    times = []
    for _ in range(cfg.inference_repeats):
        before = synchronized_time(device)
        result = sliding_window(model, images, available, cfg, device)
        times.append(synchronized_time(device)-before)
        windows = result["runtime"]["windows"]
        del result
    return dict(status="measured", subject=subject, warmup_volumes=1,
                repeated_seconds=times, median_seconds=float(np.median(times)),
                p90_seconds=float(np.quantile(times, .9)), volume_shape=list(images.shape[1:]),
                patch=list(cfg.patch), windows=windows, includes_cpu_reconstruction=True,
                peak_allocated_bytes=torch.cuda.max_memory_allocated(device) if device.type == "cuda" else None,
                peak_reserved_bytes=torch.cuda.max_memory_reserved(device) if device.type == "cuda" else None)

**`apply_condition`** — Same seed and damage for all variants, independent of execution order.

In [ ]:
def apply_condition(images, subject, condition, seed):
    """Same seed and damage for all variants, independent of execution order."""
    flags = np.asarray(condition["available"], np.float32)
    result = np.asarray(images).copy()
    corruption = condition["corruption"]
    if corruption:
        digest = hashlib.sha256(f"{subject}/{condition['id']}/{seed}".encode()).digest()
        rng = np.random.default_rng(int.from_bytes(digest[:8], "little"))
        modality = corruption["modality"]
        assert flags[modality], "Cannot corrupt an absent modality."
        kwargs = {k: v for k, v in corruption.items() if k != "modality"}
        result[modality], _ = corrupt_image(result[modality], rng=rng, **kwargs)
    result *= flags[:, None, None, None]
    return result, flags

In [ ]:
def descriptive_values(values):
    values = np.asarray(values, float)
    if not values.size:
        return dict(count=0, mean=None, median=None, q05=None, q95=None, histogram_counts=[], histogram_edges=[])
    counts, edges = np.histogram(values, bins=20)
    return dict(count=values.size, mean=float(values.mean()), median=float(np.median(values)),
                q05=float(np.quantile(values, .05)), q95=float(np.quantile(values, .95)),
                histogram_counts=counts.tolist(), histogram_edges=edges.tolist())

In [ ]:
def safe_spearman(x, y):
    if len(x) < 2 or np.ptp(x) == 0 or np.ptp(y) == 0:
        return None
    value = float(spearmanr(x, y).statistic)
    return value if np.isfinite(value) else None

**`contribution_analysis`** — Utility is measured with held-out labels AFTER inference, never a model input.

In [ ]:
def contribution_analysis(output, truth, images, flags, subject, experiment, condition, split, threshold=1e-5):
    """Utility is measured with held-out labels AFTER inference, never a model input."""
    base, contributions, logits = output["base"], output["contributions"], output["logits"]
    labels = torch.from_numpy(np.asarray(truth).astype(np.int64))
    final_ce = F.cross_entropy(logits[None], labels[None], reduction="none")[0]
    candidate_contributions = F.pad(output["corrections"], (0, 0, 0, 0, 0, 0, 1, 0)) * torch.as_tensor(flags)[:, None, None, None, None]
    candidate_logits = base + candidate_contributions.sum(0)
    candidate_ce = F.cross_entropy(candidate_logits[None], labels[None], reduction="none")[0]
    domain = np.any(images[flags.astype(bool)] != 0, axis=0)
    masks = dict(brain=domain, background=domain & (truth == 0), **{k: np.isin(truth, v) for k, v in REGIONS.items()})
    gate_rows, correction_rows = [], []
    for modality in np.flatnonzero(flags):
        without_ce = F.cross_entropy((logits - contributions[modality])[None], labels[None], reduction="none")[0]
        candidate_without = F.cross_entropy((candidate_logits - candidate_contributions[modality])[None], labels[None], reduction="none")[0]
        delta = (without_ce - final_ce).numpy()
        candidate_delta = (candidate_without - candidate_ce).numpy()
        q = output["gates"][modality, 0].numpy()
        r = output["corrections"][modality].numpy()
        c = contributions[modality, 1:].numpy()
        energy = output["detail_energy"][modality, 0].numpy()
        for region, mask in masks.items():
            common = dict(subject=subject, experiment=experiment, condition=condition, split=split,
                          modality=MODALITIES[modality], region=region, voxel_count=int(mask.sum()))
            # Deterministic stride sampling bounds correlation cost; summaries use all voxels.
            indices = np.flatnonzero(mask)[::max(1, int(mask.sum()) // 20000)]
            gate_rows.append(common | descriptive_values(q[mask]) | dict(
                measured_gated_utility_mean=float(delta[mask].mean()) if mask.any() else None,
                helpful_fraction=float((delta[mask] > 0).mean()) if mask.any() else None,
                gate_utility_spearman=safe_spearman(q.ravel()[indices], delta.ravel()[indices]),
                candidate_utility_spearman=safe_spearman(q.ravel()[indices], candidate_delta.ravel()[indices]),
                energy_utility_spearman=safe_spearman(energy.ravel()[indices], delta.ravel()[indices]),
                interpretation="window-averaged gate versus volume-level CE removal; not calibrated quality"))
            correction_rows.append(common | dict(
                absolute_candidate_mean=float(np.abs(r[:, mask]).mean()) if mask.any() else None,
                absolute_contribution_mean=float(np.abs(c[:, mask]).mean()) if mask.any() else None,
                signed_contribution_mean=c[:, mask].mean(1).tolist() if mask.any() else None,
                nonnegligible_fraction=float((np.abs(c[:, mask]).max(0) > threshold).mean()) if mask.any() else None,
                threshold=threshold, class_order=["NCR_NET", "edema", "ET"]))
    bounds, stable = stability_bound(base, contributions, threshold)
    pred, initial = logits.argmax(0).numpy(), base.argmax(0).numpy()
    stable = stable.numpy()
    stability = []
    for region, mask in masks.items():
        n = int(mask.sum())
        stable_mask = mask & stable
        base_error = mask & (initial != truth)
        stability.append(dict(subject=subject, experiment=experiment, condition=condition, split=split,
                              region=region, voxels=n, stable_fraction=float(stable[mask].mean()) if n else None,
                              stable_accuracy=float((pred[stable_mask] == truth[stable_mask]).mean()) if stable_mask.any() else None,
                              repaired_base_errors=int((base_error & (pred == truth)).sum()),
                              base_errors=int(base_error.sum()), introduced_errors=int((mask & (initial == truth) & (pred != truth)).sum()),
                              stable_wrong=int((stable_mask & (pred != truth)).sum()),
                              stable_false_positive=int((stable_mask & (truth == 0) & (pred > 0)).sum()),
                              stable_false_negative=int((stable_mask & (truth > 0) & (pred == 0)).sum())))
    indices = np.flatnonzero(domain)[::max(1, int(domain.sum()) // 2048)]
    check = verify_stability_enumeration(base.reshape(4, -1)[:, indices], contributions.reshape(4, 4, -1)[:, :, indices])
    return gate_rows, correction_rows, stability, check

**`evaluate_condition`** — Evaluate only an explicitly permitted held-out subject list.

In [ ]:
def evaluate_condition(model, store, subjects, split_manifest, split, condition, experiment, cfg, device,
                       stage="C", analysis=False, deadline=None, previous=None, save_progress=None):
    """Evaluate only an explicitly permitted held-out subject list."""
    assert split in ("validation", "test")
    assert set(subjects) <= set(split_manifest[split])
    assert not set(subjects) & set(split_manifest["train"])
    if split == "test":
        assert cfg.run_final_test and cfg.protocol_locked
    previous = previous or {}
    rows, gates, corrections = [list(previous.get(k, [])) for k in ("rows", "gates", "corrections")]
    stability, checks, timings, losses = [list(previous.get(k, [])) for k in ("stability", "stability_checks", "timings", "losses")]
    completed = list(previous.get("completed_subjects", []))
    assert set(completed) <= set(subjects) and len(completed) == len(set(completed))
    def snapshot():
        return dict(status="completed" if len(completed) == len(subjects) else "partial",
                    completed_subjects=completed, requested_subjects=list(subjects),
                    rows=rows, gates=gates, corrections=corrections, stability=stability,
                    stability_checks=checks, timings=timings, losses=losses,
                    mean_loss=float(np.mean(losses)) if losses else None)
    for subject in subjects:
        if subject in completed:
            continue
        if deadline is not None and time.perf_counter() >= deadline:
            break
        images, truth, metadata = store.load(subject)
        damaged, flags = apply_condition(images, subject, condition, cfg.split_seed)
        output = sliding_window(model, damaged, flags, cfg, device, stage, analysis)
        prediction = output["logits"].argmax(0).numpy()
        patient_rows = subject_metrics(prediction, truth, metadata["spacing"], subject, experiment, condition["id"], split, cfg.surface_tolerance_mm)
        et_volume = float((truth == 3).sum() * np.prod(metadata["spacing"]))
        size_group = "absent" if not et_volume else "not_stratified" if cfg.small_et_threshold_mm3 is None else "small" if et_volume <= cfg.small_et_threshold_mm3 else "larger"
        for row in patient_rows:
            row.update(et_volume_mm3=et_volume, et_size_group=size_group, small_et_threshold_mm3=cfg.small_et_threshold_mm3)
        rows += patient_rows
        loss = segmentation_loss(output["logits"][None], torch.from_numpy(np.asarray(truth).astype(np.int64))[None])
        losses.append(float(loss))
        timings.append(dict(subject=subject, experiment=experiment, condition=condition["id"], split=split, **output["runtime"]))
        if analysis:
            g, c, s, check = contribution_analysis(output, truth, damaged, flags, subject, experiment, condition["id"], split, cfg.nonzero_threshold)
            gates += g; corrections += c; stability += s
            checks.append(dict(subject=subject, experiment=experiment, **check))
            for modality in range(4):
                if flags[modality]:
                    removed = (output["logits"] - output["contributions"][modality]).argmax(0).numpy()
                    rows += subject_metrics(removed, truth, metadata["spacing"], subject, experiment,
                                            condition["id"] + f"_remove_detail_{MODALITIES[modality]}", split, cfg.surface_tolerance_mm)
            rows += subject_metrics(output["base"].argmax(0).numpy(), truth, metadata["spacing"], subject, experiment,
                                    condition["id"] + "_base_only", split, cfg.surface_tolerance_mm)
        completed.append(subject)
        if save_progress:
            save_progress(snapshot())
        print(f"{experiment} {split} {condition['id']}: {len(completed)}/{len(subjects)} subjects", flush=True)
        del output
    return snapshot()

**`validation_score`** — Same patient/region Dice selection, without unused surface or lesion metrics.

In [ ]:
def validation_score(model, store, subjects, split, cfg, device, stage):
    """Same patient/region Dice selection, without unused surface or lesion metrics."""
    assert subjects and set(subjects) <= set(split["validation"])
    assert not set(subjects) & set(split["train"])
    scores, losses = [], []
    for index, subject in enumerate(subjects, start=1):
        images, truth, _ = store.load(subject)
        output = sliding_window(model, images, np.ones(4, np.float32), cfg, device, stage)
        prediction = output["logits"].argmax(0).numpy()
        for classes in REGIONS.values():
            predicted_region, true_region = np.isin(prediction, classes), np.isin(truth, classes)
            denominator = int(predicted_region.sum()) + int(true_region.sum())
            scores.append(2 * int((predicted_region & true_region).sum()) / denominator if denominator else 1.)
        labels = torch.from_numpy(np.asarray(truth).astype(np.int64))[None]
        losses.append(float(segmentation_loss(output["logits"][None], labels)))
        del output
        if index % 10 == 0 or index == len(subjects):
            print(f"Validation: {index}/{len(subjects)} subjects", flush=True)
    return float(np.mean(scores)), float(np.mean(losses))

## G — GPU training, checkpointing and resumption
    CUDA uses float16 autocast and gradient scaling; losses reduce in float32.
    Checkpoint large blocks and frozen Stage C candidates to manage memory.
    No width reduction or automatic patch-size change is permitted. Latest checkpoints
    preserve optimizer/scheduler/scaler/RNG state; best checkpoints use validation
    macro Dice. Pause at the time cap and resume from attached outputs.

**`SessionBudgetReached`** — The latest checkpoint is saved; continue with the same output directory.

In [ ]:
class SessionBudgetReached(RuntimeError):
    """The latest checkpoint is saved; continue with the same output directory."""

In [ ]:
def configure_stage(model, stage, variant):
    for parameter in model.parameters():
        parameter.requires_grad_(False)
    model.freeze_candidates = stage == "C" and not variant["joint"]
    if stage in ("A", "B", "RAW") or variant["joint"]:
        for parameter in model.backbone.parameters():
            parameter.requires_grad_(True)
        if model.splitter:
            for parameter in model.splitter.parameters():
                parameter.requires_grad_(True)
    if stage == "B" or (stage == "C" and variant["joint"]):
        for parameter in model.refiner.parameters():
            parameter.requires_grad_(True)
    if stage == "C":
        if model.gate:
            for parameter in model.gate.parameters():
                parameter.requires_grad_(True)
        if model.global_logits is not None:
            model.global_logits.requires_grad_(True)
    assert any(p.requires_grad for p in model.parameters()), "This stage has no trainable parameters."

In [ ]:
def make_optimizer(model, stage, updates, cfg):
    lr = cfg.gate_lr if stage == "C" else cfg.lr
    optimizer = torch.optim.AdamW([p for p in model.parameters() if p.requires_grad], lr=lr, weight_decay=cfg.weight_decay)
    warmup = min(cfg.warmup_updates, max(1, updates // 10))
    def multiplier(step):
        if step < warmup:
            return (step + 1) / warmup
        progress = min(1., (step - warmup) / max(1, updates - warmup))
        return 0.5 * (1 + np.cos(np.pi * progress))
    scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, multiplier)
    return optimizer, scheduler

In [ ]:
def random_states():
    return dict(python=random.getstate(), numpy=np.random.get_state(), torch=torch.get_rng_state(),
                cuda=torch.cuda.get_rng_state_all() if torch.cuda.is_available() else [])

In [ ]:
def restore_random_states(states):
    random.setstate(states["python"])
    np.random.set_state(states["numpy"])
    torch.set_rng_state(states["torch"].cpu())
    if torch.cuda.is_available() and states["cuda"]:
        torch.cuda.set_rng_state_all([s.cpu() for s in states["cuda"]])

**`save_checkpoint`** — Full checkpoints are trusted local artifacts; never load arbitrary pickles.

In [ ]:
def save_checkpoint(path, model, optimizer, scheduler, scaler, stage, update, score, best_score,
                    cfg, variant, split, history, fingerprint, stale=0, elapsed=0.):
    """Full checkpoints are trusted local artifacts; never load arbitrary pickles."""
    path = Path(path)
    payload = dict(format_version=1, model_state=model.state_dict(), architecture=model.architecture_config,
                   optimizer_state=optimizer.state_dict(), scheduler_state=scheduler.state_dict(),
                   scaler_state=scaler.state_dict(), stage=stage, update=update,
                   epoch=(update + cfg.epoch_updates - 1) // cfg.epoch_updates,
                   validation_score=score, best_validation_score=best_score,
                   selection_criterion="mean patient Dice across WT,TC,ET on clean validation",
                   seed=cfg.seed, config=asdict(cfg), variant=variant, split_manifest=split,
                   history=history, rng=random_states(), fingerprint=fingerprint,
                   stale_validations=stale, elapsed_training_seconds=elapsed)
    tmp = path.with_suffix(".tmp")
    torch.save(payload, tmp)
    tmp.replace(path)

In [ ]:
def load_checkpoint(path, model, device, optimizer=None, scheduler=None, scaler=None, fingerprint=None):
    checkpoint_data = torch.load(path, map_location=device, weights_only=False)
    if fingerprint is not None and checkpoint_data["fingerprint"] != fingerprint:
        raise ValueError("Resume configuration/split mismatch; use a new output directory.")
    model.load_state_dict(checkpoint_data["model_state"], strict=True)
    if optimizer is not None:
        optimizer.load_state_dict(checkpoint_data["optimizer_state"])
        scheduler.load_state_dict(checkpoint_data["scheduler_state"])
        scaler.load_state_dict(checkpoint_data["scaler_state"])
        restore_random_states(checkpoint_data["rng"])
    return checkpoint_data

**`train_update`** — One optimizer update; no hidden gradient accumulation or clipping.

In [ ]:
def train_update(model, batch, optimizer, scaler, stage, cfg, variant, device):
    """One optimizer update; no hidden gradient accumulation or clipping."""
    model.train()
    images = batch["images"].to(device, non_blocking=True)
    labels = batch["labels"].to(device, non_blocking=True)
    available = batch["available"].to(device, non_blocking=True)
    optimizer.zero_grad(set_to_none=True)
    with autocast_context(device, cfg.precision):
        outputs = model(images, available, stage="A" if stage == "RAW" else stage)
    # Explicitly outside autocast: CE, softmax, pooling and reductions are float32.
    loss, terms = stage_objective(outputs, labels, stage, cfg, variant, batch["resolution"])
    if not bool(torch.isfinite(loss)):
        raise FloatingPointError("Nonfinite training loss")
    scaler.scale(loss).backward()
    scaler.step(optimizer)
    scaler.update()
    return terms

In [ ]:
def training_fingerprint(cfg, variant, split, stage, updates, subjects):
    keys = ("patch", "batch_size", "precision", "seed", "lr", "gate_lr", "weight_decay",
            "branch_dropout", "utility_tau", "utility_weight", "corruption_probability",
            "warmup_updates", "validation_every", "val_subjects", "synthetic")
    payload = dict(config={k: getattr(cfg, k) for k in keys}, variant=variant,
                   split=split["sha256"], stage=stage, updates=updates, subjects=subjects)
    return hashlib.sha256(json.dumps(safe_json(payload), sort_keys=True).encode()).hexdigest()

**`run_training_stage`** — Resume latest; select best validation weights separately from last update.

In [ ]:
def run_training_stage(model, store, subjects, validation_subjects, split, cfg, variant,
                       stage, updates, directory, device, deadline, stream_offset=0):
    """Resume latest; select best validation weights separately from last update."""
    directory = Path(directory)
    directory.mkdir(parents=True, exist_ok=True)
    latest, best = directory / "latest.pt", directory / "best.pt"
    configure_stage(model, stage, variant)
    optimizer, scheduler = make_optimizer(model, stage, updates, cfg)
    scaler = torch.amp.GradScaler("cuda", enabled=device.type == "cuda" and cfg.precision == "amp_fp16")
    fingerprint = training_fingerprint(cfg, variant, split, stage, updates, subjects)
    completed, history, best_score, stale, prior_seconds = 0, [], -float("inf"), 0, 0.
    if latest.exists():
        state = load_checkpoint(latest, model, device, optimizer, scheduler, scaler, fingerprint)
        completed, history = state["update"], state["history"]
        best_score, stale, prior_seconds = state["best_validation_score"], state["stale_validations"], state["elapsed_training_seconds"]
        print(f'Resuming {stage}: {completed}/{updates} updates already saved; {max(0, updates-completed)} remaining.', flush=True)
    already_stopped = bool(cfg.early_stopping_patience and stale >= cfg.early_stopping_patience)
    dataset = PatchDataset(store, subjects, cfg.patch, cfg.seed, 0 if already_stopped else max(0, updates-completed),
                           cfg.corruption_probability if variant["corruption"] else 0., start=stream_offset + completed)
    loader = make_loader(dataset)
    started, saved = time.perf_counter(), time.perf_counter()
    if device.type == "cuda":
        torch.cuda.reset_peak_memory_stats(device)
    score = None
    for batch in loader:
        if time.perf_counter() >= deadline:
            save_checkpoint(latest, model, optimizer, scheduler, scaler, stage, completed, score, best_score,
                            cfg, variant, split, history, fingerprint, stale, prior_seconds + time.perf_counter()-started)
            write_json(directory / "history.json", history)
            raise SessionBudgetReached(f"Paused {stage} at {completed}/{updates}; resume {latest}")
        before = synchronized_time(device)
        terms = train_update(model, batch, optimizer, scaler, stage, cfg, variant, device)
        scheduler.step()
        completed += 1
        seconds = synchronized_time(device) - before
        row = dict(stage=stage, update=completed, epoch=(completed-1)//cfg.epoch_updates+1,
                   subject=batch["subject"][0], patch_seconds=seconds,
                   patches_per_second=1/seconds, learning_rate=optimizer.param_groups[0]["lr"], **terms)
        validate_now = completed % cfg.validation_every == 0 or completed == updates
        if validate_now:
            print(f"{directory.parent.name} {stage} {completed}/{updates}: validating {len(validation_subjects)} subjects", flush=True)
            val_start = synchronized_time(device)
            score, val_loss = validation_score(model, store, validation_subjects, split, cfg, device,
                                                "A" if stage == "RAW" else stage)
            row.update(validation_dice=score, validation_loss=val_loss,
                       validation_seconds=synchronized_time(device)-val_start)
            improved = score > best_score
            best_score, stale = max(score, best_score), 0 if improved else stale+1
        history.append(row)
        elapsed = prior_seconds + time.perf_counter() - started
        if validate_now or time.perf_counter()-saved >= cfg.checkpoint_seconds:
            save_checkpoint(latest, model, optimizer, scheduler, scaler, stage, completed, score, best_score,
                            cfg, variant, split, history, fingerprint, stale, elapsed)
            write_json(directory / "history.json", history)
            saved = time.perf_counter()
            if validate_now and improved:
                save_checkpoint(best, model, optimizer, scheduler, scaler, stage, completed, score, best_score,
                                cfg, variant, split, history, fingerprint, stale, elapsed)
        if completed % cfg.epoch_updates == 0 or validate_now:
            print(f"{directory.parent.name} {stage} {completed}/{updates}: loss={terms['total']:.4f}, best_val={best_score:.4f}", flush=True)
        if cfg.early_stopping_patience and stale >= cfg.early_stopping_patience:
            break
    if not best.exists():
        raise RuntimeError("No validation-selected checkpoint exists; complete at least one validation.")
    state = load_checkpoint(best, model, device)
    result = dict(status="completed" if completed == updates else "early_stopped", stage=stage,
                  updates_completed=completed, updates_planned=updates, history=history,
                  checkpoint=str(best), checkpoint_sha256=file_hash(best), checkpoint_bytes=best.stat().st_size,
                  selected_update=state["update"], selected_epoch=state["epoch"], validation_score=state["validation_score"],
                  training_seconds=prior_seconds + time.perf_counter()-started,
                  peak_allocated_bytes=torch.cuda.max_memory_allocated(device) if device.type == "cuda" else None,
                  peak_reserved_bytes=torch.cuda.max_memory_reserved(device) if device.type == "cuda" else None,
                  parameter_counts=parameter_counts(model))
    write_json(directory / "stage_result.json", result)
    return result

In [ ]:
def train_stage_a(*args, **kwargs):
    return run_training_stage(*args, stage="A", **kwargs)

In [ ]:
def train_stage_b(*args, **kwargs):
    return run_training_stage(*args, stage="B", **kwargs)

In [ ]:
def train_stage_c(*args, **kwargs):
    return run_training_stage(*args, stage="C", **kwargs)

**`profile_training_stages`** — Disposable full models; measured optimizer steps include fetching each patch.

In [ ]:
def profile_training_stages(store, subjects, cfg, variant, device):
    """Disposable full models; measured optimizer steps include fetching each patch."""
    rows = []
    for stage in ("A", "B", "C"):
        seed_everything(cfg.seed)
        model = build_model(variant, cfg.checkpoint_blocks).to(device)
        configure_stage(model, stage, variant)
        total = cfg.profile_warmup + cfg.profile_updates
        optimizer, _ = make_optimizer(model, stage, total, cfg)
        scaler = torch.amp.GradScaler("cuda", enabled=device.type == "cuda" and cfg.precision == "amp_fp16")
        dataset = PatchDataset(store, subjects, cfg.patch, cfg.seed, total)
        iterator, times = iter(make_loader(dataset)), []
        if device.type == "cuda":
            torch.cuda.reset_peak_memory_stats(device)
        for index in range(total):
            before = synchronized_time(device)
            batch = next(iterator)
            train_update(model, batch, optimizer, scaler, stage, cfg, variant, device)
            elapsed = synchronized_time(device)-before
            if index >= cfg.profile_warmup:
                times.append(elapsed)
        rows.append(dict(stage=stage, warmup=cfg.profile_warmup, measured_updates=len(times),
                         median_step_seconds=float(np.median(times)), p90_step_seconds=float(np.quantile(times, .9)),
                         peak_allocated_bytes=torch.cuda.max_memory_allocated(device) if device.type == "cuda" else None,
                         peak_reserved_bytes=torch.cuda.max_memory_reserved(device) if device.type == "cuda" else None,
                         device=str(device), patch=list(cfg.patch), includes_loading=True))
        del optimizer, model, scaler
        if device.type == "cuda":
            torch.cuda.empty_cache()
    return rows

## H — Experiment execution
    The current selection is B5 only, running A (base), B (corrections), then C
    (utility gates). These stages are parts of one model, not ablation experiments.
    The registry controls every variant. B0 and B1 train for the full update budget.
    B2/B3/B4/B5/A1/A6 reuse the same A/B candidate checkpoint, with matched gate
    initialization for B4/B5/A1. B2 gets an extra B-style refinement interval so
    the total optimizer-update count matches. This does not imply equal GPU cost.
    Secondary ablations are fully executable; unselected runs receive `not_run`.

**`copy_candidate_weights`** — Strict per-module copies preserve shared Stage B candidates across gate controls.

In [ ]:
def copy_candidate_weights(target, source_checkpoint, device):
    """Strict per-module copies preserve shared Stage B candidates across gate controls."""
    source = torch.load(source_checkpoint, map_location=device, weights_only=False)
    state = source["model_state"]
    for name in ("backbone", "splitter", "refiner", "gate"):
        module = getattr(target, name)
        if module is not None:
            prefix = name + "."
            child = {k[len(prefix):]: v for k, v in state.items() if k.startswith(prefix)}
            module.load_state_dict(child, strict=True)

**`train_candidate`** — Run A then B; restoring each best checkpoint is part of the declared protocol.

In [ ]:
def train_candidate(model, store, train_ids, val_ids, split, cfg, variant, folder, device, deadline):
    """Run A then B; restoring each best checkpoint is part of the declared protocol."""
    stages = []
    for stage, updates, offset in (("A", cfg.updates_a, 0), ("B", cfg.updates_b, cfg.updates_a)):
        stages.append(run_training_stage(model, store, train_ids, val_ids, split, cfg, variant,
                                         stage, updates, Path(folder)/stage, device, deadline, offset))
    return stages

**`evaluate_experiment`** — One JSON per condition supports interrupted robustness sweeps.

In [ ]:
def evaluate_experiment(model, store, subjects, split, partition, experiment, cfg, device, folder, checkpoint_hash, deadline):
    """One JSON per condition supports interrupted robustness sweeps."""
    condition_results, condition_status = [], []
    for condition in evaluation_conditions(cfg, final=partition == "test"):
        path = Path(folder)/partition/(condition["id"] + ".json")
        expected = dict(subjects=subjects, condition=condition, checkpoint_hash=checkpoint_hash,
                        split_hash=split["sha256"], patch=list(cfg.patch), surface_tolerance=cfg.surface_tolerance_mm,
                        small_et_threshold_mm3=cfg.small_et_threshold_mm3)
        result = None
        if path.exists():
            saved = json.loads(path.read_text())
            if saved["identity"] != expected:
                raise ValueError("Existing evaluation belongs to a different configuration/checkpoint.")
            result = saved["result"]
        needs_evaluation = result is None or result.get("status", "completed") != "completed"
        if needs_evaluation and time.perf_counter() < deadline:
            def save_progress(partial_result):
                write_json(path, dict(identity=expected, result=partial_result))
            result = evaluate_condition(model, store, subjects, split, partition, condition, experiment,
                                        cfg, device, analysis=condition["id"] == "clean_m15",
                                        deadline=deadline, previous=result, save_progress=save_progress)
            write_json(path, dict(identity=expected, result=result))
        elif result is None:
            condition_status.append(dict(condition=condition, status="not_run", reason="session time cap; resume evaluation"))
            continue
        condition_results.append(result)
        condition_status.append(dict(condition=condition, status=result.get("status", "completed"),
            completed_subjects=result.get("completed_subjects", subjects),
            reason=None if result.get("status", "completed") == "completed" else "time cap; resume remaining patients"))
    return dict(status="completed" if all(c["status"] == "completed" for c in condition_status) else "partial",
                conditions=condition_status, results=condition_results)

**`run_experiment_sweep`** — Run only selected variants; default B5 still trains all three required stages.

In [ ]:
def run_experiment_sweep(store, split, cfg, device):
    """Run only selected variants; default B5 still trains all three required stages."""
    registry = experiment_registry()
    root = Path(cfg.output_dir)/"experiments"
    root.mkdir(parents=True, exist_ok=True)
    statuses = {f"{variant}_s{seed}": dict(variant_id=variant, seed=seed, description=definition["description"],
                 configuration=definition, status="not_run", reason="not selected in this allocation")
                for seed in cfg.seeds for variant, definition in registry.items()}
    started = time.perf_counter()
    deadline = started + cfg.max_hours * 3600
    training_deadline = started + cfg.max_hours * 3600 * cfg.training_budget_fraction
    train_ids = split["train"][:min(cfg.train_subjects, len(split["train"]))]
    val_ids = split["validation"][:cfg.val_subjects]
    for seed in cfg.seeds:
        seeded = replace(cfg, seed=seed)
        shared_checkpoint, shared_stages = None, []
        for variant_id in cfg.variants:
            experiment = f"{variant_id}_s{seed}"
            variant = registry[variant_id]
            folder = root/experiment
            folder.mkdir(exist_ok=True)
            record = statuses[experiment]
            record.update(reason=None, exact_run_configuration=asdict(seeded), train_subjects=train_ids,
                          validation_subjects=val_ids, split_hash=split["sha256"], evidence_level=cfg.mode,
                          scientific_comparison_status="preliminary; assess convergence and repeated seeds")
            status_path = folder/"result.json"
            seed_everything(seed)
            model = None
            try:
                if time.perf_counter() >= deadline:
                    record.update(status="not_run", reason="session allocation exhausted")
                    continue
                model = build_model(variant, cfg.checkpoint_blocks).to(device)
                record["parameters"] = parameter_counts(model)
                stages = []
                if variant["gate"] == "raw":
                    stages.append(run_training_stage(model, store, train_ids, val_ids, split, seeded, variant,
                                  "RAW", cfg.updates_a+cfg.updates_b+cfg.updates_c, folder/"RAW", device, training_deadline))
                else:
                    if variant_id in ("B2", "B3", "B4", "B5", "A1", "A6"):
                        if shared_checkpoint is None:
                            seed_everything(seed)  # Independent of which gate variant runs first.
                            candidate = build_model(registry["B5"], cfg.checkpoint_blocks).to(device)
                            try:
                                shared_stages = train_candidate(candidate, store, train_ids, val_ids, split, seeded,
                                                registry["B5"], root/f"shared_candidates_s{seed}", device, training_deadline)
                            finally:
                                del candidate
                            shared_checkpoint = shared_stages[-1]["checkpoint"]
                        copy_candidate_weights(model, shared_checkpoint, device)
                        stages = [dict(s, shared=True) for s in shared_stages]
                        record["shared_stage_b_checkpoint"] = shared_checkpoint
                        # Same gate initialization and patch stream for B4/B5/A1.
                        seed_everything(seed + 20000)
                    else:
                        stages = train_candidate(model, store, train_ids, val_ids, split, seeded, variant,
                                                  folder, device, training_deadline)
                    # Give ungated B2 the same total number of updates, without changing gates.
                    stage = "B" if variant_id == "B2" else "C"
                    stages.append(run_training_stage(model, store, train_ids, val_ids, split, seeded, variant,
                                   stage, cfg.updates_c, folder/"refinement", device, training_deadline,
                                   cfg.updates_a+cfg.updates_b))
                selected = stages[-1]
                record.update(status="trained", stages=stages, checkpoint=selected["checkpoint"],
                              checkpoint_sha256=selected["checkpoint_sha256"],
                              validation_selection_score=selected["validation_score"],
                              training_seconds=sum(s["training_seconds"] for s in stages),
                              incremental_training_seconds=sum(s["training_seconds"] for s in stages if not s.get("shared")))
                write_json(status_path, record)
                if variant_id == "B5":
                    # Retain the usable trained model even if evaluation spans sessions.
                    export_best_detailtrust(statuses, cfg.output_dir, device)
                benchmark_path = folder/"inference_benchmark.json"
                benchmark_identity = dict(checkpoint_sha256=selected["checkpoint_sha256"],
                    subject=val_ids[0], patch=list(cfg.patch), precision=cfg.precision,
                    repeats=cfg.inference_repeats, device=str(device))
                if benchmark_path.exists():
                    benchmark_saved = json.loads(benchmark_path.read_text())
                    if benchmark_saved.get("identity") != benchmark_identity:
                        raise ValueError("Inference benchmark configuration changed; use a fresh output directory.")
                    record["inference_benchmark"] = benchmark_saved["result"]
                elif time.perf_counter() < deadline:
                    record["inference_benchmark"] = benchmark_inference(model, store, val_ids[0], cfg, device)
                    write_json(benchmark_path, dict(identity=benchmark_identity, result=record["inference_benchmark"]))
                else:
                    record["inference_benchmark"] = dict(status="not_run", reason="time cap")
                record["validation"] = evaluate_experiment(model, store, val_ids, split, "validation", experiment,
                                                           cfg, device, folder, selected["checkpoint_sha256"], deadline)
                if cfg.run_final_test:
                    record["test"] = evaluate_experiment(model, store, split["test"], split, "test", experiment,
                                                         cfg, device, folder, selected["checkpoint_sha256"], deadline)
                else:
                    record["test"] = dict(status="not_run", reason="test remains sealed; enable only with locked main protocol")
                record["status"] = "completed" if record["validation"]["status"] == "completed" and record["test"]["status"] in ("completed", "not_run") else "evaluation_partial"
            except SessionBudgetReached as error:
                record.update(status="paused", reason=str(error))
            except torch.cuda.OutOfMemoryError:
                record.update(status="failed", reason="GPU OOM; no automatic architecture change. Profile checkpointing or explicitly use matched 80³ patches.")
            except Exception as error:
                record.update(status="failed", reason=f"{type(error).__name__}: {error}")
            finally:
                if record["status"] in ("paused", "failed"):
                    record["partial_history"] = {str(p.relative_to(root)): json.loads(p.read_text())
                        for directory in (folder, root/f"shared_candidates_s{seed}") if directory.exists()
                        for p in directory.rglob("history.json")}
                if model is not None:
                    del model
                if device.type == "cuda":
                    torch.cuda.empty_cache()
                write_json(status_path, record)
                write_json(Path(cfg.output_dir)/"ablation_results.json", statuses)
    return statuses

**`export_best_detailtrust`** — Choose B5 seed by validation only. No untrained checkpoint placeholders.

In [ ]:
def export_best_detailtrust(statuses, output_dir, device):
    """Choose B5 seed by validation only. No untrained checkpoint placeholders."""
    candidates = [r for r in statuses.values() if r["variant_id"] == "B5" and r.get("checkpoint")]
    if not candidates:
        return dict(status="not_produced", reason="B5 has not completed its full A/B/C training sequence; inspect saved stage checkpoints")
    selected = max(candidates, key=lambda r: r["validation_selection_score"])
    source = selected["checkpoint"]
    destination = Path(output_dir)/"best_detailtrust_model.pt"
    shutil.copy2(source, destination)
    state = torch.load(destination, map_location="cpu", weights_only=False)
    rebuilt = DetailTrustNet(**state["architecture"])
    rebuilt.load_state_dict(state["model_state"], strict=True)
    config = dict(architecture=state["architecture"], variant_id="B5", seed=selected["seed"],
                  label_mapping={"0": 0, "1": 1, "2": 2, "4": 3}, modality_order=list(("t1", "t1ce", "t2", "flair")),
                  preprocessing="RAS; verified 1mm; nonzero z-score; available-image crop; 4-voxel halo")
    write_json(Path(output_dir)/"best_detailtrust_model_config.json", config)
    return dict(status="saved", path=str(destination), sha256=file_hash(destination), size_bytes=destination.stat().st_size,
                selected_epoch=state["epoch"], selected_update=state["update"], seed=selected["seed"],
                validation_score=state["validation_score"], criterion=state["selection_criterion"],
                architecture=state["architecture"], format="PyTorch full resume checkpoint v1",
                strict_state_dict_load="passed", source_experiment=f"B5_s{selected['seed']}")

## Paper figure exports
Measured training and patient results are exported as PNG (300 DPI), PDF and SVG, with captions and source records. Missing evaluations are recorded, never plotted as successes. Validation and test data are separate. Journal-specific formatting remains to be checked.

In [ ]:
matplotlib.use('Agg')

**`save_paper_figure`** — Export 300-DPI PNG plus vector PDF/SVG and a caption/source-data sidecar.

In [ ]:
def save_paper_figure(fig, directory, name, caption, source):
    """Export 300-DPI PNG plus vector PDF/SVG and a caption/source-data sidecar."""
    directory = Path(directory)
    directory.mkdir(parents=True, exist_ok=True)
    paths = []
    with plt.rc_context({'pdf.fonttype': 42, 'svg.fonttype': 'none', 'font.size': 9}):
        for suffix in ('png', 'pdf', 'svg'):
            path = directory/f'{name}.{suffix}'
            fig.savefig(path, dpi=300, facecolor='white')
            paths.append(str(path))
    plt.close(fig)
    write_json(directory/f'{name}.json', dict(caption=caption, source=source,
               formats=paths, journal_requirements='Not specified; verify target journal before submission'))
    return dict(status='saved', files=paths, caption=caption)

**`distribution_panel`** — Each dot is a measured patient record; undefined values are counted explicitly.

In [ ]:
def distribution_panel(ax, rows, metric, categories, category_key, ylabel, bounded=False):
    """Each dot is a measured patient record; undefined values are counted explicitly."""
    for index, category in enumerate(categories, 1):
        group = [r for r in rows if r[category_key] == category]
        values = [r[metric] for r in group if r.get(metric) is not None and np.isfinite(r[metric])]
        if values:
            ax.boxplot([values], positions=[index], widths=.45, showfliers=False,
                       medianprops={'color': '#D55E00'})
            ax.scatter(index + np.linspace(-.16, .16, len(values)), values, s=9,
                       color='#0072B2', alpha=.65)
        ax.text(index, 1.01, f'n={len(values)}\nNA={len(group)-len(values)}',
                transform=ax.get_xaxis_transform(), ha='center', fontsize=7)
    ax.set_xticks(range(1, len(categories)+1), categories, rotation=25)
    ax.set_ylabel(ylabel)
    if bounded:
        ax.set_ylim(0, 1)
    ax.grid(axis='y', alpha=.2)

**`plot_training_panels`** — Separate stages because their objectives and update indices differ.

In [ ]:
def plot_training_panels(history, directory):
    """Separate stages because their objectives and update indices differ."""
    artifacts = {}
    for experiment in sorted({r['experiment'] for r in history}):
        rows = [r for r in history if r['experiment'] == experiment]
        stages = list(dict.fromkeys(r['stage'] for r in rows))
        for name, keys, label in (
            ('training_objectives', ('total', 'segmentation', 'base', 'branch_auxiliary', 'utility'), 'Loss (stage-specific)'),
            ('validation_trajectory', ('validation_dice',), 'Validation patient macro Dice'),
            ('learning_rate', ('learning_rate',), 'Learning rate')):
            fig, axes = plt.subplots(1, len(stages), figsize=(max(5, 4*len(stages)), 3.8), layout='constrained', squeeze=False)
            for ax, stage in zip(axes[0], stages):
                group = [r for r in rows if r['stage'] == stage]
                for key in keys:
                    values = [r for r in group if r.get(key) is not None]
                    if values:
                        ax.plot([r['update'] for r in values], [r[key] for r in values],
                                label=key, lw=.7, marker='o' if key == 'validation_dice' else None, markersize=3)
                ax.set(title=f'{experiment}: Stage {stage}', xlabel='Optimizer update within stage', ylabel=label)
                if name == 'validation_trajectory':
                    ax.set_ylim(0, 1)
                ax.legend(fontsize=7); ax.grid(alpha=.2)
            caption = ('Raw logged values without smoothing. Stages have separate axes/objectives. '
                       'Validation points are measured checkpoints only; lines guide the eye. '
                       f'Incomplete training records: {any(r.get("incomplete", False) for r in rows)}.')
            artifacts[f'{experiment}_{name}'] = save_paper_figure(fig, directory, f'{experiment}_{name}', caption,
                [{k: r[k] for k in ('stage', 'update', *keys) if k in r} for r in rows])
    return artifacts

In [ ]:
def plot_accuracy_panels(rows, directory, prefix):
    clean = [r for r in rows if r['condition'] == 'clean_m15' and r['region'] in REGIONS]
    artifacts = {}
    if not clean:
        return artifacts
    for name, metrics in (
        ('overlap', [('dice', 'Dice'), ('iou', 'IoU')]),
        ('boundaries', [('hd95_mm', 'HD95 (mm)'), ('surface_dice', 'Surface Dice')]),
        ('detection', [('precision', 'Voxel precision'), ('recall', 'Voxel sensitivity')])):
        fig, axes = plt.subplots(1, 2, figsize=(9, 4.5), layout='constrained')
        for ax, (metric, label) in zip(axes, metrics):
            distribution_panel(ax, clean, metric, list(REGIONS), 'region', label, metric != 'hd95_mm')
        fig.suptitle(prefix + ': clean four-modality evaluation')
        caption = ('Dots are patients, boxes show quartiles and median, whiskers use 1.5 IQR. '
                   'NA counts undefined metrics; empty-case handling is in metrics_per_subject.csv. '
                   'Surface Dice uses the saved physical tolerance and unweighted boundary voxel centres. '
                   'These distributions describe patients, not training-seed uncertainty.')
        artifacts[name] = save_paper_figure(fig, directory, prefix+'_'+name, caption, clean)
    et = [r for r in clean if r['region'] == 'ET' and r.get('et_volume_mm3', 0) > 0]
    if et:
        fig, ax = plt.subplots(figsize=(6, 4), layout='constrained')
        ax.scatter([r['et_volume_mm3']/1000 for r in et], [r['dice'] for r in et], color='#0072B2', marker='o')
        ax.set(xscale='log', xlabel='Reference ET volume (mL, log scale)', ylabel='ET Dice', ylim=(0, 1), title=prefix)
        artifacts['et_size'] = save_paper_figure(fig, directory, prefix+'_et_size',
            f'One point per ET-positive patient (n={len(et)}); ET-absent patients excluded from the log-volume axis only. No fitted trend.', et)
    return artifacts

**`plot_robustness_panels`** — Availability heatmap shows counts; corruption curves use matched patients.

In [ ]:
def plot_robustness_panels(rows, directory, prefix):
    """Availability heatmap shows counts; corruption curves use matched patients."""
    artifacts = {}
    selected = [r for r in rows if re.fullmatch(r'clean_m\d{2}', r['condition']) and r['region'] in REGIONS]
    if selected:
        matrix = np.full((3, 15), np.nan)
        counts = np.zeros((3, 15), int)
        for i, region in enumerate(REGIONS):
            for bits in range(1, 16):
                values = [r['dice'] for r in selected if r['region'] == region and r['condition'] == f'clean_m{bits:02}']
                if values:
                    matrix[i, bits-1], counts[i, bits-1] = np.mean(values), len(values)
        fig, ax = plt.subplots(figsize=(13, 4.8), layout='constrained')
        im = ax.imshow(matrix, vmin=0, vmax=1, cmap=plt.get_cmap('viridis').with_extremes(bad='#cccccc'), aspect='auto')
        labels = ['+'.join(MODALITIES[m] for m in range(4) if bits & (1 << m)) for bits in range(1, 16)]
        ax.set_xticks(range(15), labels, rotation=65, ha='right'); ax.set_yticks(range(3), list(REGIONS))
        for i in range(3):
            for j in range(15):
                text = f'{matrix[i,j]:.2f}\nn={counts[i,j]}' if counts[i,j] else 'not run'
                ax.text(j, i, text, ha='center', va='center', fontsize=7, color='black' if not counts[i,j] or matrix[i,j] > .55 else 'white')
        fig.colorbar(im, ax=ax, label='Patient macro Dice'); ax.set_title(prefix + ': modality availability')
        artifacts['missing_modalities'] = save_paper_figure(fig, directory, prefix+'_missing_modalities',
            'Each cell averages measured patients for that region/subset; n is shown. Gray means not evaluated. Unequal n indicates incomplete evaluation; do not interpret such cells as matched comparisons.', selected)
    damaged = [r for r in rows if not r['condition'].startswith('clean') and r['region'] in REGIONS]
    if damaged:
        conditions = sorted({r['condition'] for r in damaged})
        fig, axes = plt.subplots(1, 3, figsize=(13, 4.5), layout='constrained')
        sources = []
        for ax, region in zip(axes, REGIONS):
            for i, condition in enumerate(conditions):
                bits = re.search(r'_m(\d+)', condition).group(1)
                clean = {r['subject']: r['dice'] for r in rows if r['region'] == region and r['condition'] == 'clean_m'+bits}
                paired = [r for r in damaged if r['region'] == region and r['condition'] == condition and r['subject'] in clean]
                delta = [r['dice'] - clean[r['subject']] for r in paired]
                if delta:
                    ax.scatter(i + np.linspace(-.12, .12, len(delta)), delta, s=8, color='#0072B2')
                    ax.plot(i, np.mean(delta), 'D', color='#D55E00')
                sources += [dict(r, matched_clean_dice=clean[r['subject']]) for r in paired]
            ax.axhline(0, color='black', lw=.7); ax.set_xticks(range(len(conditions)), conditions, rotation=75, ha='right')
            ax.set(title=region, ylabel='Corrupted minus matched clean Dice')
        artifacts['corruption'] = save_paper_figure(fig, directory, prefix+'_corruption',
            'Dots are paired patient differences with identical modality availability; diamonds are their means. No unmatched patients are included. Simulated corruptions do not establish clinical robustness.', sources)
    return artifacts

**`plot_interpretability_panels`** — No mixing of patients across splits, seeds, regions, or corruption settings.

In [ ]:
def plot_interpretability_panels(records, directory, prefix, experiment, partition):
    """No mixing of patients across splits, seeds, regions, or corruption settings."""
    artifacts = {}
    def select(key):
        return [r for r in records[key] if r['experiment'] == experiment and r['split'] == partition
                and r['condition'] == 'clean_m15' and r['region'] == 'brain']
    gates, corrections, stability = select('gates'), select('corrections'), select('stability')
    if gates:
        fig, axes = plt.subplots(1, 2, figsize=(10, 4.5), layout='constrained')
        distribution_panel(axes[0], gates, 'mean', MODALITIES, 'modality', 'Mean brain gate per patient', True)
        for modality, marker in zip(MODALITIES, ('o', 's', '^', 'D')):
            group = [r for r in gates if r['modality'] == modality and r.get('measured_gated_utility_mean') is not None]
            axes[1].scatter([r['mean'] for r in group], [r['measured_gated_utility_mean'] for r in group], marker=marker, label=modality, s=15)
        axes[1].axhline(0, color='black', lw=.6)
        axes[1].set(xlabel='Mean brain gate per patient', ylabel='CE without correction minus CE with correction', xlim=(0, 1))
        axes[1].legend(fontsize=8)
        artifacts['gate_utility'] = save_paper_figure(fig, directory, prefix+'_gate_utility',
            'Patient summaries on brain voxels. Positive CE difference indicates benefit in the fixed inference context. Gate scores are not calibrated probabilities; this plot is not causal evidence.', gates)
    if corrections:
        fig, axes = plt.subplots(1, 2, figsize=(10, 4.5), layout='constrained')
        distribution_panel(axes[0], corrections, 'absolute_contribution_mean', MODALITIES, 'modality', 'Mean absolute foreground logit contribution')
        for k, label in enumerate(('NCR/NET', 'edema', 'ET')):
            for m, modality in enumerate(MODALITIES):
                vals = [r['signed_contribution_mean'][k] for r in corrections if r['modality'] == modality and r.get('signed_contribution_mean') is not None]
                if vals:
                    axes[1].scatter(np.full(len(vals), m+(k-1)*.2), vals, s=10, marker=('o','s','^')[k], label=label if m == 0 else None)
        axes[1].axhline(0, color='black', lw=.7); axes[1].set_xticks(range(4), MODALITIES)
        axes[1].set_ylabel('Signed mean logit contribution per patient'); axes[1].legend(fontsize=8)
        artifacts['contributions'] = save_paper_figure(fig, directory, prefix+'_contributions',
            'One value per patient/modality (and class on the right), using the independently blended additive contributions. Magnitude alone does not imply benefit.', corrections)
    if stability:
        fig, axes = plt.subplots(1, 2, figsize=(10, 4), layout='constrained')
        valid = [r for r in stability if r.get('stable_accuracy') is not None]
        axes[0].scatter([r['stable_fraction'] for r in valid], [r['stable_accuracy'] for r in valid], s=15)
        axes[0].set(xlabel='Conditionally stable brain-voxel fraction', ylabel='Accuracy within stable voxels', xlim=(0, 1), ylim=(0, 1))
        axes[1].scatter([r['repaired_base_errors'] for r in stability], [r['introduced_errors'] for r in stability], s=15)
        axes[1].set(xlabel='Repaired base errors (voxels/patient)', ylabel='Introduced errors (voxels/patient)')
        artifacts['stability'] = save_paper_figure(fig, directory, prefix+'_stability',
            'Each point is one patient. Stability is conditional on fixed base logits and correction maps; it is not correctness or robustness to re-encoding a missing modality. Raw counts depend on brain volume.', stability)
    return artifacts

In [ ]:
def plot_detail_removal(rows, directory, prefix):
    clean = {(r['subject'], r['region']): r['dice'] for r in rows if r['condition'] == 'clean_m15' and r['region'] in REGIONS}
    conditions = ['clean_m15_base_only'] + ['clean_m15_remove_detail_'+m for m in MODALITIES]
    selected = [r for r in rows if r['condition'] in conditions and (r['subject'], r['region']) in clean]
    if not selected:
        return {}
    fig, axes = plt.subplots(1, 3, figsize=(12, 4.5), layout='constrained')
    source = []
    for ax, region in zip(axes, REGIONS):
        for i, condition in enumerate(conditions):
            group = [r for r in selected if r['region'] == region and r['condition'] == condition]
            changes = [clean[(r['subject'], region)] - r['dice'] for r in group]
            ax.scatter(i + np.linspace(-.12, .12, len(changes)), changes, s=8)
            source += [dict(r, full_model_dice=clean[(r['subject'], region)]) for r in group]
        ax.axhline(0, color='black', lw=.6); ax.set_xticks(range(5), ['all', *MODALITIES], rotation=35)
        ax.set(title=region, xlabel='Detail corrections disabled', ylabel='Full minus disabled Dice')
    return {'detail_removal': save_paper_figure(fig, directory, prefix+'_detail_removal',
            'Paired patient Dice differences after disabling frozen contributions. Positive means the full correction improved Dice. This is an inference intervention, not a retrained ablation.', source)}

**`plot_ablation_comparisons`** — Display multiple measured variants on their shared clean patients only.

In [ ]:
def plot_ablation_comparisons(rows, directory):
    """Display multiple measured variants on their shared clean patients only."""
    artifacts = {}
    for partition in sorted({r['split'] for r in rows}):
        clean = [r for r in rows if r['split'] == partition and r['condition'] == 'clean_m15' and r['region'] in REGIONS]
        experiments = sorted({r['experiment'] for r in clean})
        if len(experiments) < 2:
            continue
        patients = set.intersection(*({r['subject'] for r in clean if r['experiment'] == ex} for ex in experiments))
        matched = [r for r in clean if r['subject'] in patients]
        if not matched:
            continue
        fig, axes = plt.subplots(1, 3, figsize=(max(10, len(experiments)*2), 4.5), layout='constrained')
        for ax, region in zip(axes, REGIONS):
            distribution_panel(ax, [r for r in matched if r['region'] == region], 'dice', experiments, 'experiment', region+' Dice', True)
        artifacts[partition+'_variant_comparison'] = save_paper_figure(fig, directory, partition+'_variant_comparison',
            'Measured variants on intersecting clean patients only. One dot per patient/model, not independent training replicates. Check training budgets and experiment status before treating this as a controlled ablation.', matched)
    return artifacts or {'variant_comparison': dict(status='not_produced', reason='At least two measured variants with matched patients required')}

In [ ]:
def generate_paper_figures(records, output_dir):
    directory = Path(output_dir)/'figures'
    artifacts = plot_training_panels(records['history'], directory)
    artifacts.update(plot_ablation_comparisons(records['rows'], directory))
    groups = sorted({(r['experiment'], r['split']) for r in records['rows']})
    expected = ('overlap', 'boundaries', 'detection', 'et_size', 'missing_modalities', 'corruption', 'gate_utility', 'contributions', 'stability', 'detail_removal')
    for experiment, partition in groups:
        rows = [r for r in records['rows'] if r['experiment'] == experiment and r['split'] == partition]
        prefix = re.sub(r'[^A-Za-z0-9_-]', '_', experiment+'_'+partition)
        result = plot_accuracy_panels(rows, directory, prefix)
        result.update(plot_robustness_panels(rows, directory, prefix))
        result.update(plot_interpretability_panels(records, directory, prefix, experiment, partition))
        result.update(plot_detail_removal(rows, directory, prefix))
        for name in expected:
            artifacts[prefix+'_'+name] = result.get(name, dict(status='not_produced', reason='Required measured records unavailable'))
    if not groups:
        artifacts['evaluation_figures'] = dict(status='not_produced', reason='No completed patient evaluations; training curves only')
    write_json(directory/'paper_figures_manifest.json', artifacts)
    (directory/'README.md').write_text('Figure candidates for manuscript preparation\n\nPNG: 300 DPI; PDF/SVG: vector graphics. Each JSON sidecar contains the caption and plotted source records.\nValidation/test and experiments are kept separate. Missing measurements are not fabricated.\nReview the run status and patient counts before publication. Journal-specific requirements remain unchecked.\n', encoding='utf-8')
    return artifacts

## I — Report and artifact generation
    JSON uses null for undefined values and includes the reason/count. CSV and
    figures contain actual records only. A missing trained checkpoint is recorded
    as missing, never replaced with random weights named 'best'.

In [ ]:
matplotlib.use("Agg")

In [ ]:
REPORT_FIELDS = ("schema_version", "project", "architecture", "architecture_equations", "implementation_deviations",
                 "environment", "hardware", "dataset", "dataset_integrity", "label_mapping", "preprocessing",
                 "data_splits", "random_seeds", "training_configuration", "training_stages", "training_history",
                 "best_checkpoint", "model_parameters", "model_complexity", "validation_metrics", "test_metrics",
                 "per_subject_metrics", "class_metrics", "region_metrics", "missing_modality_robustness",
                 "spatial_corruption_robustness", "detail_branch_contributions", "utility_gate_analysis",
                 "stability_analysis", "ablation_study", "statistical_analysis", "runtime_and_memory",
                 "reproducibility", "warnings", "limitations", "experiment_status")

In [ ]:
def implementation_decisions():
    return [
        "B3 means four trainable global modality sigmoid scalars, learned in Stage C; no image-dependent pooling network.",
        "A1 permutes targets spatially within each subject/modality; target marginal distribution is retained.",
        "A4 is one-level Haar low-band inverse reconstruction using pairwise averaging/repetition; even origin, no learned filters.",
        "A6 unfreezes all components in Stage C after the same A/B initialization; detached targets still recompute each step.",
        "B2 receives an extra B-style refinement stage of C updates, matching total update count; gate controls fork the PRE-refinement B checkpoint.",
        "Utility average pooling excludes padded positions at boundaries, an explicit resolution of unspecified pooling padding.",
        "A7 uses the declared central-patch coarse lattice and adaptive-average occupancy operator for globally resolution-degraded samples only. It does not assert equivalence to the entire blur/resample MRI simulator or local blend mask.",
        "The BraTS20 adapter reorients losslessly to RAS and rejects non-1mm or mismatched affines. It does not silently resample another dataset.",
        "Patient lists from the cited author protocol are not recovered; the notebook uses a saved custom subject split.",
        "Surface Dice uses unweighted 6-connected boundary voxel centres, not area-weighted surfels; HD95 concatenates directed distances.",
        "Stage transitions restore the best validation checkpoint; epochs are fixed update-reporting intervals, not complete dataset passes.",
        "Main/extended start at 96 cubed; CPU synthetic tests explicitly use 16 cubed while preserving every model width.",
        "Training augmentations use aligned flips without geometric interpolation; no additional rotations are required by the starting protocol.",
        "Gate and candidate maps are window averages. Their product is not used as the final contribution; c is independently blended.",
    ]

**`write_csv`** — No fake header-only metric files: return a reason if there are no records.

In [ ]:
def write_csv(path, rows):
    """No fake header-only metric files: return a reason if there are no records."""
    if not rows:
        return dict(status="not_produced", reason="No measured records")
    fields = sorted({key for row in rows for key in row})
    with open(path, "w", newline="", encoding="utf-8") as handle:
        writer = csv.DictWriter(handle, fieldnames=fields)
        writer.writeheader()
        for row in rows:
            cleaned = safe_json(row)
            writer.writerow({k: json.dumps(v, sort_keys=True) if isinstance(v, (dict, list)) else v for k, v in cleaned.items()})
    return dict(status="saved", path=str(path), rows=len(rows))

In [ ]:
def collect_results(statuses):
    records = dict(rows=[], gates=[], corrections=[], stability=[], stability_checks=[], timings=[], history=[])
    for experiment, status in statuses.items():
        if not status.get("stages"):
            for source, history in status.get("partial_history", {}).items():
                records["history"] += [dict(row, experiment=experiment, source=source, incomplete=True) for row in history]
        for stage in status.get("stages", []):
            records["history"] += [dict(row, experiment=experiment, shared=stage.get("shared", False)) for row in stage["history"]]
        for split in ("validation", "test"):
            for condition in status.get(split, {}).get("results", []):
                for key in ("rows", "gates", "corrections", "stability", "stability_checks", "timings"):
                    records[key] += condition.get(key, [])
    return records

**`generate_plots`** — Render measured patient results in PNG/PDF/SVG, with caption/data sidecars.

In [ ]:
def generate_plots(records, summary, output_dir):
    """Render measured patient results in PNG/PDF/SVG, with caption/data sidecars."""
    return generate_paper_figures(records, output_dir)

In [ ]:
def build_report(cfg, split, integrity, statuses, best_checkpoint, checks, profiles, dataset_evidence):
    records = collect_results(statuses)
    summary = aggregate_metrics(records["rows"])
    environment = environment_record(cfg.output_dir)
    comparisons = {f"{seed}_{partition}_{endpoint}": paired_patient_comparison(records["rows"], f"B5_s{seed}", f"B4_s{seed}",
                   partition=partition, endpoint=endpoint)
                   for seed in cfg.seeds for partition in ("validation", "test") for endpoint in ("overall", "degraded", "clean_ET")}
    for key, status in statuses.items():
        comparator = f"B4_s{status['seed']}" if status["variant_id"] in ("B5", "A1") else f"B1_s{status['seed']}"
        status["comparison"] = paired_patient_comparison(records["rows"], key, comparator)
        status["seed_count_for_this_record"] = 1 if status.get("checkpoint") else 0
    report = {key: None for key in REPORT_FIELDS}
    report.update(schema_version="1.0.0", project=dict(name="DetailTrust-Net", evidence_level=cfg.mode,
                  synthetic=cfg.synthetic, hypothesis="Utility-supervised removable detail corrections improve segmentation versus segmentation-only gates"),
        architecture=dict(widths=[32, 64, 128, 256, 320], modalities=list(MODALITIES), classes=4,
                          reference_parameters=16934644, correction_input_channels=37, gate_input_channels=44),
        architecture_equations=dict(split="H=X-sum softmax(alpha)*G_sigma(X)", anchor="r=R(F,H,e)-R(F,0,e)",
                 fusion="z_foreground=z0_foreground+sum a*q*r; z_background=z0_background",
                 utility="sigmoid(AvgPool3(CE(z_all-r,Y)-CE(z_all,Y))/0.1), detached",
                 stability="B_j=(z_winner-z_j)-sum_m max(0,c_m,winner-c_m,j)"),
        implementation_deviations=implementation_decisions(), environment=environment,
        hardware={k: v for k, v in environment.items() if k.startswith("gpu") or k == "cuda"},
        dataset=dataset_evidence, dataset_integrity=integrity, label_mapping=LABEL_MAPPING,
        preprocessing=dict(orientation="RAS", spacing_mm=[1, 1, 1], interpolation="lossless reorientation; labels never interpolated; image degradation uses trilinear",
                           normalization="per-modality nonzero foreground z-score", patch=cfg.patch, halo=4, cache_gb=cfg.cache_gb,
                           crop="available MRI foreground, eight-voxel margin; never labels"),
        data_splits=split, random_seeds=dict(training=cfg.seeds, split=cfg.split_seed), training_configuration=asdict(cfg),
        training_stages={k: [{a: b for a, b in s.items() if a != "history"} for s in v.get("stages", [])] for k, v in statuses.items()},
        training_history=records["history"], best_checkpoint=best_checkpoint,
        model_parameters={k: v.get("parameters") for k, v in statuses.items()},
        model_complexity=dict(macs=None, flops=None, reason="No validated profiler result; no theoretical estimate presented as measured"),
        validation_metrics=[r for r in summary if r["split"] == "validation"],
        test_metrics=dict(status="completed_or_partial" if cfg.run_final_test else "not_run", records=[r for r in summary if r["split"] == "test"]),
        per_subject_metrics=records["rows"], class_metrics=[r for r in summary if r["region"] not in ("WT", "TC", "ET")],
        region_metrics=[r for r in summary if r["region"] in ("WT", "TC", "ET")],
        missing_modality_robustness=[r for r in summary if r["condition"].startswith("clean_m") and "_remove" not in r["condition"] and "base_only" not in r["condition"]],
        spatial_corruption_robustness=[r for r in summary if not r["condition"].startswith("clean_m")],
        detail_branch_contributions=records["corrections"], utility_gate_analysis=records["gates"],
        stability_analysis=dict(records=records["stability"], enumeration_checks=records["stability_checks"],
                                scope="fixed computed contributions; not correctness or whole-input removal"),
        ablation_study=statuses, statistical_analysis=dict(paired_patient_comparisons=comparisons, requested_seed_count=len(cfg.seeds),
                 completed_seed_counts={v: sum(r.get("checkpoint") is not None for r in statuses.values() if r["variant_id"] == v) for v in experiment_registry()},
                 limitation="Single seed is preliminary; bootstrap conditions on weights. No voxel-level independent-sample inference."),
        runtime_and_memory=dict(profiles=profiles, full_volume_timings=records["timings"],
                               inference_benchmarks={k: v.get("inference_benchmark") for k, v in statuses.items()}),
        reproducibility=dict(checks=checks, json_nonfinite_policy="null, with undefined-case counts", cuda_determinism="warn_only; exact bitwise cross-device reproducibility not promised",
            resume_restore=json.loads((Path(cfg.output_dir)/'resume_restore.json').read_text()) if (Path(cfg.output_dir)/'resume_restore.json').exists() else None),
        warnings=["No improvement or novelty claim follows from a smoke/pilot run.", "A large gate or correction is not necessarily useful.",
                  "Mounted dataset licence/version must be checked against dataset card and original BraTS terms."],
        limitations=["Synthetic corruption is not clinical missingness or scanner-domain validation.",
                     "No external patient cohort is evaluated.", "Published M2FTrans/evidential/SwitchNet code is not reproduced in this notebook; internal controls alone do not support SOTA claims.",
                     "Labels are one released consensus per subject, not independent multi-rater annotations.",
                     "Patient subset/one seed results do not establish convergence or generalization."],
        experiment_status={k: dict(status=v["status"], reason=v.get("reason")) for k, v in statuses.items()})
    return report, records, summary

In [ ]:
def export_report(report, records, summary, output_dir):
    directory = Path(output_dir)
    write_json(directory/"detailtrust_report.json", report)
    write_json(directory/"training_history.json", records["history"])
    write_json(directory/"ablation_results.json", report["ablation_study"])
    write_json(directory/"stability_analysis.json", report["stability_analysis"])
    artifacts = {}
    for name, rows in (("metrics_per_subject.csv", records["rows"]), ("metrics_summary.csv", summary),
                       ("gate_analysis.csv", records["gates"]), ("correction_contributions.csv", records["corrections"])):
        artifacts[name] = write_csv(directory/name, rows)
    artifacts["figures"] = generate_plots(records, summary, directory)
    parsed = json.loads((directory/"detailtrust_report.json").read_text())
    assert set(REPORT_FIELDS) <= set(parsed)
    write_json(directory/"artifact_manifest.json", artifacts)
    return artifacts

In [ ]:
def verify_artifacts(output_dir):
    names = ["best_detailtrust_model.pt", "best_detailtrust_model_config.json", "detailtrust_report.json",
             "training_history.json", "ablation_results.json", "split_manifest.json", "experiment_config.json",
             "metrics_per_subject.csv", "metrics_summary.csv", "gate_analysis.csv", "correction_contributions.csv",
             "stability_analysis.json", "README.md"]
    result = {name: dict(exists=(Path(output_dir)/name).is_file(),
                        bytes=(Path(output_dir)/name).stat().st_size if (Path(output_dir)/name).is_file() else None)
              for name in names}
    write_json(Path(output_dir)/"artifact_verification.json", result)
    return result

## J — Tests before expensive training
    Synthetic tensors test geometry-independent model algebra and gradients.
    The NIfTI test writes a tiny reoriented volume and exercises the actual adapter.
    These are correctness checks, not evidence of tumour-segmentation accuracy.

In [ ]:
def check_labels_and_splits():
    assert remap_labels(np.array([0, 1, 2, 4])).tolist() == [0, 1, 2, 3]
    try:
        remap_labels(np.array([3]))
        raise AssertionError("Unknown original labels were accepted")
    except ValueError:
        pass
    labels = torch.tensor([[[[0, 1, 2, 3]]]])
    regions = region_labels(labels)
    assert regions[0, 0].sum() == 3 and regions[0, 1].sum() == 2 and regions[0, 2].sum() == 1
    split = make_split([dict(id=f"p{i}") for i in range(369)], 9)
    assert [len(split[k]) for k in ("train", "validation", "test")] == [219, 50, 100]
    assert_split(split)
    assert split == make_split([dict(id=f"p{i}") for i in range(369)], 9)

In [ ]:
def check_gaussian_and_haar():
    x = torch.randn(1, 4, 16, 16, 16)
    for spacing in (1., 2.):
        kernel = gaussian_kernel(.8, spacing, 4, x.device, x.dtype)
        assert torch.allclose(kernel.sum(), torch.tensor(1.))
        assert torch.allclose(kernel, kernel.flip(0))
    narrow = gaussian_kernel(.8, 2., 4, x.device, x.dtype)
    wide = gaussian_kernel(.8, 1., 4, x.device, x.dtype)
    assert narrow[4] > wide[4]
    for split in (GaussianSplit(), GaussianSplit(False), HaarSplit()):
        low, high = split(x)
        torch.testing.assert_close(low + high, x)
    split = GaussianSplit()
    assert split.alpha.shape == (4, 3)
    torch.testing.assert_close(split.alpha[0], torch.tensor([-1., 1., -1.]))
    low, high = split(x)
    low.square().mean().backward()
    assert torch.isfinite(split.alpha.grad).all() and split.alpha.grad.abs().sum() > 0

In [ ]:
def check_availability_and_anchor():
    x = torch.randn(1, 4, 16, 16, 16)
    a = torch.tensor([[1., 0., 1., 0.]])
    masked, maps = availability_maps(x, a)
    assert not masked[:, 1].any() and not maps[:, 3].any()
    try:
        availability_maps(x, torch.zeros_like(a))
        raise AssertionError("All-missing case was accepted")
    except AssertionError as error:
        assert "At least one" in str(error)
    features = torch.randn(1, 32, 8, 8, 8)
    identity = modality_identity(features, 2)
    detail = torch.zeros(1, 1, 8, 8, 8)
    correction = DetailCorrection()(features, detail, identity)
    assert torch.count_nonzero(correction) == 0
    gate = SpatialUtilityGate()(features, detail, correction, identity, torch.ones(1, 4, 8, 8, 8))
    assert gate.shape == (1, 1, 8, 8, 8) and bool(((gate >= 0) & (gate <= 1)).all())

In [ ]:
def check_fusion_and_stability():
    base = torch.randn(1, 4, 4, 4, 4)
    corrections = torch.randn(1, 4, 3, 4, 4, 4)
    gates = torch.rand(1, 4, 1, 4, 4, 4)
    available = torch.tensor([[1., 0., 1., 1.]])
    logits, contributions = fuse_logits(base, corrections, gates, available)
    torch.testing.assert_close(logits[:, 0], base[:, 0])
    assert torch.count_nonzero(contributions[:, 1]) == 0
    torch.testing.assert_close(logits, base + contributions.sum(1))
    for bits in itertools.product((0., 1.), repeat=4):
        retention = torch.tensor([bits])
        kept, _ = fuse_logits(base, corrections, gates, available, retention)
        expected = base + (contributions * retention[:, :, None, None, None, None]).sum(1)
        torch.testing.assert_close(kept, expected)
    result = verify_stability_enumeration(base[0], contributions[0])
    assert result["passed"], result

In [ ]:
def check_utility_sign_and_detachment():
    base = torch.zeros(1, 4, 3, 3, 3, requires_grad=True)
    correction = torch.zeros(1, 4, 3, 3, 3, 3)
    correction[:, 0, 2] = 1.  # Helps true class ET; removal must increase CE.
    correction[:, 1, 2] = -0.2  # Harms true ET in the same context.
    correction.requires_grad_()
    available = torch.ones(1, 4)
    labels = torch.full((1, 3, 3, 3), 3, dtype=torch.long)
    target, delta = utility_targets(base, correction, available, labels)
    assert bool((target[:, 0] > .5).all()) and bool((target[:, 1] < .5).all())
    assert bool((target[:, 2:] == .5).all())
    assert not target.requires_grad and not delta.requires_grad
    q = torch.full_like(target, .5, requires_grad=True)
    loss = balanced_gate_bce(q, target, labels, available)
    loss.backward()
    assert torch.isfinite(q.grad).all() and base.grad is None and correction.grad is None

In [ ]:
def check_model_and_gradients():
    model = DetailTrustNet(checkpoint_blocks=True)
    assert parameter_counts(model)["total"] == 16934644
    images = torch.randn(1, 4, 24, 24, 24)
    labels = torch.randint(0, 4, (1, 16, 16, 16))
    available = torch.ones(1, 4)
    output = model(images, available)
    assert output["logits"].shape == (1, 4, 16, 16, 16)
    assert output["corrections"].shape == (1, 4, 3, 16, 16, 16)
    assert output["gates"].shape == (1, 4, 1, 16, 16, 16)
    loss = segmentation_loss(output["logits"], labels)
    loss.backward()
    assert bool(torch.isfinite(loss))
    assert all(p.grad is None or torch.isfinite(p.grad).all() for p in model.parameters())
    for module in (model.backbone, model.splitter, model.refiner, model.gate):
        assert any(p.grad is not None and p.grad.abs().sum() > 0 for p in module.parameters())

In [ ]:
def check_frozen_stage_and_roundtrip():
    model = DetailTrustNet(checkpoint_blocks=False)
    variant = experiment_registry()["B5"]
    configure_stage(model, "C", variant)
    assert all(not p.requires_grad for m in (model.backbone, model.splitter, model.refiner) for p in m.parameters())
    assert all(p.requires_grad for p in model.gate.parameters())
    images, available = torch.randn(1, 4, 24, 24, 24), torch.ones(1, 4)
    output = model(images, available)
    assert not output["corrections"].requires_grad and output["gates"].requires_grad
    with tempfile.TemporaryDirectory() as temporary:
        path = Path(temporary)/"state.pt"
        torch.save(model.state_dict(), path)
        rebuilt = DetailTrustNet(checkpoint_blocks=False)
        rebuilt.load_state_dict(torch.load(path, weights_only=True), strict=True)
        with torch.no_grad():
            torch.testing.assert_close(output["logits"], rebuilt(images, available)["logits"])

In [ ]:
def check_patch_stream_and_metrics():
    store = SyntheticStore()
    subjects = list(store.records)[:4]
    dataset = PatchDataset(store, subjects, (16, 16, 16), 22, 10)
    batch = dataset[3]
    resumed = PatchDataset(store, subjects, (16, 16, 16), 22, 7, start=3)[0]
    for key in ("images", "labels", "available", "resolution"):
        torch.testing.assert_close(batch[key], resumed[key])
    assert batch["images"].shape == (4, 24, 24, 24)
    assert batch["labels"].shape == (16, 16, 16)
    empty = np.zeros((5, 5, 5), bool)
    assert binary_metrics(empty, empty, (1, 1, 1))["dice"] == 1
    single = empty.copy(); single[2, 2, 2] = True
    assert binary_metrics(single, empty, (1, 1, 1))["hd95_mm"] is None
    shifted = np.roll(single, 1, axis=0)
    assert binary_metrics(single, shifted, (2, 1, 1))["hd95_mm"] == 2
    assert lesion_metrics(single, single)["lesion_recall"] == 1
    for family, severity in (("resolution", 2.), ("noise", .1), ("bias", .1), ("ghost", .2), ("ringing", .2)):
        image, _, _ = store.load(subjects[0])
        damaged, _ = corrupt_image(image[0], family, severity, np.random.default_rng(1))
        assert damaged.shape == image[0].shape and np.isfinite(damaged).all()

In [ ]:
def check_sliding_identity_and_leakage():
    store = SyntheticStore(size=16)
    cfg = configuration("smoke", patch=(16, 16, 16), synthetic=True)
    images, labels, _ = store.load("synthetic_000")
    model = DetailTrustNet(checkpoint_blocks=False)
    result = sliding_window(model, images, np.ones(4, np.float32), cfg, torch.device("cpu"), analysis=True)
    assert result["logits"].shape == (4, 16, 16, 16)
    torch.testing.assert_close(result["logits"], result["base"] + result["contributions"].sum(0))
    split = make_split([dict(id=s) for s in store.records], 5)
    try:
        evaluate_condition(model, store, [split["train"][0]], split, "validation", {}, "test", cfg, torch.device("cpu"))
        raise RuntimeError("Training subject entered held-out evaluation")
    except AssertionError:
        pass

In [ ]:
def check_json_and_registry():
    with tempfile.TemporaryDirectory() as temporary:
        path = Path(temporary)/"report.json"
        write_json(path, dict(undefined=float("nan"), infinity=float("inf"), count=np.int64(3)))
        assert json.loads(path.read_text()) == dict(undefined=None, infinity=None, count=3)
    registry = experiment_registry()
    assert set(registry) == {"B0", "B1", "B2", "B3", "B4", "B5", "A1", "A2", "A3", "A4", "A5", "A6", "A7"}
    assert registry["B4"]["utility"] is False and registry["B5"]["utility"] is True
    for key, variant in registry.items():
        model = build_model(variant, checkpoint_blocks=False)
        with torch.no_grad():
            output = model(torch.randn(1, 4, 24, 24, 24), torch.tensor([[1., 0., 1., 1.]]))
        assert output["logits"].shape == (1, 4, 16, 16, 16)
        assert torch.isfinite(output["logits"]).all()
        assert torch.count_nonzero(output["contributions"][:, 1]) == 0

**`check_nifti_adapter`** — Actual NIfTI read/reorientation test when nibabel is installed.

In [ ]:
def check_nifti_adapter():
    """Actual NIfTI read/reorientation test when nibabel is installed."""
    if importlib.util.find_spec("nibabel") is None:
        return dict(status="blocked", reason="nibabel not installed in local CPU environment")
    import nibabel as nib
    with tempfile.TemporaryDirectory() as temporary:
        root = Path(temporary)
        subject = "BraTS20_Training_001"
        affine = np.diag([-1., 1., 1., 1.]); affine[0, 3] = 15
        for name in ("t1", "t1ce", "t2", "flair", "seg"):
            array = np.zeros((16, 16, 16), np.float32)
            array[2:9, 2:9, 2:9] = 4 if name == "seg" else 2
            nib.save(nib.Nifti1Image(array, affine), root/f"{subject}_{name}.nii.gz")
        records, _ = discover_subjects(root)
        images, labels, meta = load_subject(records[0])
        assert images.shape == (4, 16, 16, 16) and labels.max() == 3 and meta["orientation"] == "RAS"
    return dict(status="passed")

**`check_optimizer_resume`** — A saved Stage B update resumes to identical weights and stochastic loss.

In [ ]:
def check_optimizer_resume():
    """A saved Stage B update resumes to identical weights and stochastic loss."""
    cfg = configuration("smoke", patch=(16, 16, 16), synthetic=True, precision="float32")
    variant = experiment_registry()["B5"]
    seed_everything(91)
    model = build_model(variant, checkpoint_blocks=False)
    configure_stage(model, "B", variant)
    optimizer, scheduler = make_optimizer(model, "B", 3, cfg)
    scaler = torch.amp.GradScaler("cuda", enabled=False)
    store = SyntheticStore(size=16)
    subjects = list(store.records)
    split = make_split([dict(id=s) for s in subjects], cfg.split_seed)
    batches = list(make_loader(PatchDataset(store, subjects[:2], cfg.patch, cfg.seed, 2)))
    train_update(model, batches[0], optimizer, scaler, "B", cfg, variant, torch.device("cpu"))
    scheduler.step()
    with tempfile.TemporaryDirectory() as temporary:
        path = Path(temporary)/"resume.pt"
        save_checkpoint(path, model, optimizer, scheduler, scaler, "B", 1, .0, .0,
                        cfg, variant, split, [], "resume_test")
        expected = train_update(model, batches[1], optimizer, scaler, "B", cfg, variant, torch.device("cpu"))
        scheduler.step()
        restored = build_model(variant, checkpoint_blocks=False)
        configure_stage(restored, "B", variant)
        opt2, scheduler2 = make_optimizer(restored, "B", 3, cfg)
        scaler2 = torch.amp.GradScaler("cuda", enabled=False)
        load_checkpoint(path, restored, torch.device("cpu"), opt2, scheduler2, scaler2, "resume_test")
        actual = train_update(restored, batches[1], opt2, scaler2, "B", cfg, variant, torch.device("cpu"))
        scheduler2.step()
        assert expected == actual
        assert scheduler.state_dict() == scheduler2.state_dict()
        for name, value in model.state_dict().items():
            torch.testing.assert_close(value, restored.state_dict()[name], rtol=0, atol=0)

**`check_validation_selection_equivalence`** — Fast checkpoint selection matches full evaluation and avoids costly metrics.

In [ ]:
def check_validation_selection_equivalence():
    """Fast checkpoint selection matches full evaluation and avoids costly metrics."""
    from unittest.mock import patch
    store = SyntheticStore(size=16)
    split = make_split([dict(id=s) for s in store.records], 5)
    cfg = configuration("smoke", patch=(16, 16, 16), synthetic=True, precision="float32")
    model = DetailTrustNet(checkpoint_blocks=False)
    subjects = split["validation"][:1]
    clean = dict(id="clean_m15", available=[1, 1, 1, 1], corruption=None)
    reference = evaluate_condition(model, store, subjects, split, "validation", clean,
                                   "selection", cfg, torch.device("cpu"))
    expected = np.mean([r["dice"] for r in reference["rows"] if r["region"] in ("WT", "TC", "ET")])
    # Also works in the standalone notebook, where functions share one namespace.
    with patch.dict(validation_score.__globals__, subject_metrics=lambda *a, **k: (_ for _ in ()).throw(AssertionError("Unused full metrics called"))):
        score, loss = validation_score(model, store, subjects, split, cfg, torch.device("cpu"), "C")
    assert score == expected and loss == reference["mean_loss"]
    try:
        validation_score(model, store, split["train"][:1], split, cfg, torch.device("cpu"), "C")
        raise RuntimeError("Training subject entered validation")
    except AssertionError:
        pass

In [ ]:
def run_unit_checks():
    checks = [check_labels_and_splits, check_gaussian_and_haar, check_availability_and_anchor,
              check_fusion_and_stability, check_utility_sign_and_detachment, check_model_and_gradients,
              check_frozen_stage_and_roundtrip, check_patch_stream_and_metrics,
              check_sliding_identity_and_leakage, check_json_and_registry, check_optimizer_resume,
              check_validation_selection_equivalence, check_nifti_adapter]
    results = []
    seed_everything(7)
    for check in checks:
        started = time.perf_counter()
        value = check()
        result = dict(name=check.__name__, seconds=time.perf_counter()-started, **(value or {"status": "passed"}))
        results.append(result)
        print(f"{check.__name__}: {result['status']}", flush=True)
    return results

**`tiny_overfit_check`** — Repeated one-patch B0 optimization; diagnostic only, never an evaluation score.

In [ ]:
def tiny_overfit_check(store, subjects, cfg, device, updates=20):
    """Repeated one-patch B0 optimization; diagnostic only, never an evaluation score."""
    variant = experiment_registry()["B0"]
    model = build_model(variant, cfg.checkpoint_blocks).to(device)
    configure_stage(model, "RAW", variant)
    optimizer, scheduler = make_optimizer(model, "RAW", updates, cfg)
    scaler = torch.amp.GradScaler("cuda", enabled=device.type == "cuda" and cfg.precision == "amp_fp16")
    batch = next(iter(make_loader(PatchDataset(store, subjects[:2], cfg.patch, cfg.seed, 1, 0.))))
    losses = []
    for _ in range(updates):
        losses.append(train_update(model, batch, optimizer, scaler, "RAW", cfg, variant, device)["total"])
        scheduler.step()
    return dict(status="completed", losses=losses, loss_decreased=losses[-1] < losses[0],
                interpretation="Repeated training patch sanity check, not held-out performance")

In [ ]:
checks = run_unit_checks()
write_json(output_dir / "unit_checks.json", checks)
assert all(item["status"] == "passed" for item in checks), "Resolve blocked tests before Kaggle training."

In [ ]:
overfit_check = tiny_overfit_check(store, train_ids, cfg, device, updates=5 if cfg.mode == "smoke" else 20)
write_json(output_dir / "tiny_overfit_check.json", overfit_check)
print("Repeated patch loss decreased:", overfit_check["loss_decreased"])
# A short run need not fully memorize the patch. Inspect this curve if training fails.

In [ ]:
profiles = profile_training_stages(store, train_ids, cfg, experiment_registry()["B5"], device)
write_json(output_dir / "compute_profile.json", profiles)
print(profiles)
estimated_hours = sum(n * p["median_step_seconds"] for n, p in zip(
    (cfg.updates_a, cfg.updates_b, cfg.updates_c), profiles)) / 3600
print("Measured-step extrapolation for ONE B5 run, excluding validation:", round(estimated_hours, 2), "hours")

### Run the selected experiments
This cell can run for hours. It checkpoints frequently and records paused/failed variants. Resume in a later session; do not treat a partial sweep as completed evidence.

In [ ]:
statuses = run_experiment_sweep(store, split, cfg, device)
print({name: item["status"] for name, item in statuses.items()})

In [ ]:
best_checkpoint = export_best_detailtrust(statuses, output_dir, device)
print(best_checkpoint)

In [ ]:
report, records, summary = build_report(cfg, split, integrity, statuses, best_checkpoint, checks, profiles, dataset_evidence)
artifacts = export_report(report, records, summary, output_dir)
print("Report:", output_dir / "detailtrust_report.json")

In [ ]:
if best_checkpoint["status"] == "saved":
    state = torch.load(best_checkpoint["path"], map_location=device, weights_only=False)
    best_model = DetailTrustNet(**state["architecture"]).to(device)
    best_model.load_state_dict(state["model_state"], strict=True)
    chosen_subject = val_ids[0]  # prespecified, never chosen for a flattering result
    shown_images, shown_labels, shown_metadata = store.load(chosen_subject)
    shown_output = sliding_window(best_model, shown_images, np.ones(4, np.float32), cfg, device, analysis=True)
    plot_corrections(shown_images, shown_labels, shown_output, output_dir / "figures" / "example_corrections.png")
    qualitative_figures = plot_qualitative_paper(shown_images, shown_labels, shown_output,
        output_dir / "figures", chosen_subject, shown_metadata['spacing'])
    write_json(output_dir / 'figures' / 'qualitative_manifest.json', qualitative_figures)
    del best_model, shown_output, state
    if device.type == "cuda": torch.cuda.empty_cache()

In [ ]:
runtime_readme = """DetailTrust-Net run artifacts

Read detailtrust_report.json first: experiment_status distinguishes completed,
paused, failed and not_run experiments. Smoke and synthetic results demonstrate
execution only. Metrics are patient macro averages with explicit empty cases.
No test result exists unless RUN_FINAL_TEST and PROTOCOL_LOCKED were enabled.

To resume: attach this output as a Kaggle input, set RESUME_DIRECTORY to its
detailtrust folder and retain the same dataset, split, mode and training settings.
Run the complete notebook. It restores latest optimizer/RNG checkpoints and
separately selects best validation weights. /kaggle/working is session storage;
commit notebook outputs or download them before the session is discarded.

The local launcher automates continuation: .\kaggle.ps1 --mode main --run
It attaches a private hash-verified checkpoint bundle. --fresh deliberately
starts new weights. --download excludes cache before transfer.
Paper figures: figures/paper_figures_manifest.json and qualitative_manifest.json.
Each exported figure has PNG/PDF/SVG and a caption/source-data sidecar.

To load trusted weights after running the model definition cells:
state = torch.load('best_detailtrust_model.pt', map_location='cpu', weights_only=False)
model = DetailTrustNet(**state['architecture'])
model.load_state_dict(state['model_state'], strict=True)
model.eval()
"""
(output_dir / "README.md").write_text(runtime_readme, encoding="utf-8")

In [ ]:
artifact_verification = verify_artifacts(output_dir)
for name, result in artifact_verification.items():
    print(name, "saved" if result["exists"] else "NOT PRODUCED — inspect experiment status")

In [ ]:
with open(output_dir / "detailtrust_report.json", encoding="utf-8") as handle:
    saved_report = json.load(handle)
assert set(REPORT_FIELDS) <= set(saved_report)
print("Evidence level:", saved_report["project"]["evidence_level"])
print("Best checkpoint:", saved_report["best_checkpoint"])
print("Final test:", saved_report["test_metrics"]["status"])
print("Experiments:", saved_report["experiment_status"])

## How to interpret the run
    A smoke test establishes execution only. A pilot is preliminary and may be
    undertrained. Inspect B5 versus B4, clean ET, contribution activity, repaired
    and introduced errors, and stable wrong predictions. Do not use a high stability
    fraction as evidence that the segmentation is correct.

    Keep test evaluation disabled until all methods, budgets and conditions are
    frozen. Main experiments may span multiple Kaggle quotas. Three seeds and
    external validation remain necessary for stronger claims. The notebook does
    not reproduce the published M²FTrans, evidential fusion or SwitchNet systems;
    its internal ablations alone cannot support state-of-the-art claims.

    Sources: [BraTS protocol and citations](https://www.med.upenn.edu/cbica/brats2020/data.html),
    [NiBabel orientation](https://nipy.org/nibabel/image_orientation.html),
    [PyTorch checkpointing](https://docs.pytorch.org/docs/stable/checkpoint.html),
    [Kaggle GPU guidance](https://www.kaggle.com/docs/efficient-gpu-usage).